# VGG, worked through

[*Very Deep Convolutional Networks for Large-Scale Image Recognition*](https://arxiv.org/abs/1409.1556) — Simonyan & Zisserman, ICLR 2015.

This is the notebook the study was done in, in the order it was done. The code and every output below
are exactly as they ran — on a Colab A100 for the GPU work, on Imagenette and then on all 50,000
ILSVRC-2012 validation images. Nothing has been re-executed and nothing has been tidied away, including
the runs that produced NaN, the prediction that was wrong by a factor of thirty, and the configuration
bug that only an invariant caught.

The commentary between the cells was added afterwards. The cells were not.

Clean, runnable versions of this work live beside it: [`vgg.py`](vgg.py) for the architectures,
[`cost.py`](cost.py), [`init_study.py`](init_study.py), [`dense_eval.py`](dense_eval.py) and
[`analysis.py`](analysis.py) for the measurements.

---

## 1. Shapes, before anything else

Everything downstream depends on knowing exactly what size comes out of a convolution, so that is
checked against the formula first rather than assumed.

In [1]:
import torch, torch.nn as nn

for (k, p, s) in [(3,1,1), (3,0,1), (7,3,1), (11,2,4), (2,0,2)]:
  c = nn.Conv2d(3,8, kernel_size=k, padding=p, stride=s)
  out = c(torch.zeros(1,3,224,224))
  formula = (224 + 2*p -k) // s + 1
  print(f"k={k} p={p} s={s}  ->  torch: {out.shape[-1]:>3}   formula: {formula:>3}")

k=3 p=1 s=1  ->  torch: 224   formula: 224
k=3 p=0 s=1  ->  torch: 222   formula: 222
k=7 p=3 s=1  ->  torch: 224   formula: 224
k=11 p=2 s=4  ->  torch:  55   formula:  55
k=2 p=0 s=2  ->  torch: 112   formula: 112


In [2]:
import torch, torch.nn as nn
x = torch.randn(1,3,224,224)
print("input: ", tuple(x.shape))

conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1, stride=1)
print("output: ", tuple(conv(x).shape))

input:  (1, 3, 224, 224)
output:  (1, 64, 224, 224)


In [3]:
bad = nn.Conv2d(3,8, kernel_size=3, padding=0, stride=1)
print("padding=0 -> ", tuple(bad(x).shape))

padding=0 ->  (1, 8, 222, 222)


In [5]:
pool = nn.MaxPool2d(kernel_size=3, stride=2)
print("after maxpool: ", tuple(pool(conv(x)).shape))

after maxpool:  (1, 64, 111, 111)


In [6]:
n = sum(p.numel() for p in conv.parameters())
print("number of parameters: ", n)

number of parameters:  1792


In [15]:
import torch, torch.nn as nn

block1 = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1, stride=1),
    nn.ReLU(inplace=True),
    nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1, stride=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2)
)

x = torch.randn(1,3,224,224)
print("input        ->", tuple(x.shape))

h = x
for layer in block1:
  h = layer(h)
  p = sum(q.numel() for q in layer.parameters())
  print(f"{layer.__class__.__name__:12s} -> {str(tuple(h.shape)):20s}  params {p:>7,}")

print("\nblock 1 total params:", f"{sum(p.numel() for p in block1.parameters()):,}")

input        -> (1, 3, 224, 224)
Conv2d       -> (1, 64, 224, 224)     params   1,792
ReLU         -> (1, 64, 224, 224)     params       0
Conv2d       -> (1, 64, 224, 224)     params  36,928
ReLU         -> (1, 64, 224, 224)     params       0
MaxPool2d    -> (1, 64, 112, 112)     params       0

block 1 total params: 38,720


The formula and PyTorch agree, `padding = k // 2` holds the size for any odd kernel, and the shape
trace confirms that in VGG only the max-pools change resolution — which is why 224 → 7 is exactly 2⁵.

---

## 2. Where the memory goes

VGG's reputation is that it is memory-hungry. Worth finding out where the memory actually sits before
trying to save any.

The obvious lever is `inplace=True` on the ReLUs — it should avoid storing a second copy of every
activation. Predicted saving at batch 32: several hundred megabytes.

In [3]:
import torch, torch.nn as nn

def make_block(inplace):
  return nn.Sequential(
      nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(inplace=inplace),
      nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=inplace),
      nn.MaxPool2d(2, stride=2),
  ).cuda()

def measure(inplace, batch=32):
  torch.cuda.empty_cache()
  torch.cuda.reset_peak_host_memory_stats()

  block = make_block(inplace)
  x = torch.randn(batch, 3, 224, 224, device="cuda")

  out = block(x)
  loss = out.sum()
  loss.backward()
  torch.cuda.synchronize()

  peak = torch.cuda.max_memory_allocated() / 1024**2
  del block, x, out, loss
  return peak

measure(True); measure(False)

a = measure(False)
b = measure(True)
print(f"inplace=False : {a:8.1f} MB")
print(f"inplace=True  : {b:8.1f} MB")
print(f"bachat       : {a-b:8.1f} MB")

inplace=False :   2080.7 MB
inplace=True  :   2080.7 MB
bachat       :      0.0 MB


**0.0 MB.** Identical to the last decimal, which rules out measurement noise.

The saving is zero. That is a real measurement and the obvious conclusions from it are both wrong: the
rewrite did take effect, and the memory was not saved.

The way to find out which is to stop looking at one number and watch memory through the whole step.

In [4]:
import torch, torch.nn as nn
MB = 1024**2

block = nn.Sequential(
    nn.Conv2d(3,  64, 3, padding=1), nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),
).cuda()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

x = torch.randn(32, 3, 224, 224, device="cuda")
print(f"{'step':26s} {'zinda MB':>10s} {'peak MB':>10s}")
print(f"{'input':26s} {torch.cuda.memory_allocated()/MB:10.1f} {torch.cuda.max_memory_allocated()/MB:10.1f}")

h = x
for i, layer in enumerate(block):
    h = layer(h)
    torch.cuda.synchronize()
    print(f"{i} {layer.__class__.__name__:24s} {torch.cuda.memory_allocated()/MB:10.1f} {torch.cuda.max_memory_allocated()/MB:10.1f}")

loss = h.sum()
loss.backward()
torch.cuda.synchronize()
print(f"{'after backward':26s} {torch.cuda.memory_allocated()/MB:10.1f} {torch.cuda.max_memory_allocated()/MB:10.1f}")

step                         zinda MB    peak MB
input                            18.5       18.5
0 Conv2d                        410.5      410.8
1 ReLU                          410.5      802.5
2 Conv2d                        802.5     1194.7
3 ReLU                          802.5     1194.7
4 MaxPool2d                    1096.5     1194.7
after backward                  116.7     2080.7


There it is. The forward peak with `inplace=False` is **802.5 MB**; the first ReLU is where it doubles
(410.5 → 802.5). But the peak over the whole step is **2080.7 MB**, and it is set *after backward*.

So the saving is real and it is in the forward pass — it is invisible in the whole-step peak because
that peak is set while the stored activations are being consumed anyway. The earlier cell measured the
one number training actually fails on, and for that number the answer genuinely is zero.

(The mechanism, separately: `ReluBackward0` saves `_saved_result`, not `_saved_self` — the backward
pass reads the ReLU's output, so overwriting its input is legal. That explains why the rewrite works.
It does not explain the zero. The staircase does.)

---

## 3. Where the parameters go

From memory to parameters. Configuration D is the famous VGG-16, and the paper's Table 2 gives it as
"138 million" — rounded, so an exact count either lands inside the rounding or it does not.

In [14]:
import torch, torch.nn as nn

features = nn.Sequential(
    # ---------- block 1 ----------  224 -> 112
    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # ---------- block 2 ----------  112 -> 56
    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
    nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # ---------- block 3 ----------  56 -> 28
    nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
    nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
    nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # ---------- block 4 ----------  28 -> 14
    nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # ---------- block 5 ----------  14 -> 7
    nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2, 2),
)

classifier = nn.Sequential(
    nn.Linear(512*7*7, 4096), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(4096, 1000),
)

vgg16 = nn.Sequential(features, nn.Flatten(), classifier)

total_params = sum(p.numel() for p in vgg16.parameters())

print("Total parameters : ", f"{total_params:,}")
print("paper (Table 2): 138 million")

x = torch.randn(1,3,224,224)
print("input  ->", tuple(x.shape))
print("output ->", tuple(vgg16(x).shape))

print()
x = torch.randn(1, 3, 224, 224)
print("input".ljust(12), "->", tuple(x.shape))
print()

h = x
block_params = 0
block_idx = 1
total_connections = 0
conv_params_total = 0
conv_connections_total = 0

for i, layer in enumerate(features):
    input_shape = h.shape
    h = layer(h)
    p = sum(q.numel() for q in layer.parameters())
    block_params += p
    name = layer.__class__.__name__
    print(f"{name:12s} -> {str(tuple(h.shape)):20s}  params {p:>7,}")

    if isinstance(layer, nn.Conv2d):
        conv_params_total += p
        connections = h.shape[2] * h.shape[3] * layer.kernel_size[0] * layer.kernel_size[1] * layer.in_channels * layer.out_channels
        conv_connections_total += connections
        total_connections += connections

    if isinstance(layer, nn.MaxPool2d):
        print(f"\nblock {block_idx} total params: {block_params:,}\n")
        block_params = 0
        block_idx += 1

h = nn.Flatten()(h)
print("Flatten".ljust(12), "->", tuple(h.shape), " ".ljust(8), "params       0")
print()

fc_params_total = 0
fc_connections_total = 0

for layer in classifier:
    input_shape = h.shape
    h = layer(h)
    p = sum(q.numel() for q in layer.parameters())
    name = layer.__class__.__name__
    print(f"{name:12s} -> {str(tuple(h.shape)):20s}  params {p:>7,}")

    if isinstance(layer, nn.Linear):
        fc_params_total += p
        connections = layer.in_features * layer.out_features
        fc_connections_total += connections
        total_connections += connections

print("\n" + "="*60)
print(f"Total Parameters:  {total_params:,}")
print(f"Total Connections: {total_connections:,} (MACs/GFLOPs estimate)")
print("-"*60)
print(f"Conv Parameters:   {conv_params_total:>15,} ({conv_params_total/total_params*100:5.2f}%)")
print(f"Conv Connections:  {conv_connections_total:>15,} ({conv_connections_total/total_connections*100:5.2f}%)")
print(f"FC Parameters:     {fc_params_total:>15,} ({fc_params_total/total_params*100:5.2f}%)")
print(f"FC Connections:    {fc_connections_total:>15,} ({fc_connections_total/total_connections*100:5.2f}%)")
print("="*60)

Total parameters :  138,357,544
paper (Table 2): 138 million
input  -> (1, 3, 224, 224)
output -> (1, 1000)

input        -> (1, 3, 224, 224)

Conv2d       -> (1, 64, 224, 224)     params   1,792
ReLU         -> (1, 64, 224, 224)     params       0
Conv2d       -> (1, 64, 224, 224)     params  36,928
ReLU         -> (1, 64, 224, 224)     params       0
MaxPool2d    -> (1, 64, 112, 112)     params       0

block 1 total params: 38,720

Conv2d       -> (1, 128, 112, 112)    params  73,856
ReLU         -> (1, 128, 112, 112)    params       0
Conv2d       -> (1, 128, 112, 112)    params 147,584
ReLU         -> (1, 128, 112, 112)    params       0
MaxPool2d    -> (1, 128, 56, 56)      params       0

block 2 total params: 221,440

Conv2d       -> (1, 256, 56, 56)      params 295,168
ReLU         -> (1, 256, 56, 56)      params       0
Conv2d       -> (1, 256, 56, 56)      params 590,080
ReLU         -> (1, 256, 56, 56)      params       0
Conv2d       -> (1, 256, 56, 56)      params 590,080

In [15]:
import torch, torch.nn as nn

# We will use the existing features and classifier definitions
# First, let's calculate the overall totals to compute percentages
total_params_all = sum(p.numel() for p in vgg16.parameters())

# Let's compute total connections dynamically
total_connections_all = 0
h = torch.randn(1, 3, 224, 224)
for layer in features:
    h = layer(h)
    if isinstance(layer, nn.Conv2d):
        total_connections_all += h.shape[2] * h.shape[3] * layer.kernel_size[0] * layer.kernel_size[1] * layer.in_channels * layer.out_channels

h = nn.Flatten()(h)
for layer in classifier:
    if isinstance(layer, nn.Linear):
        total_connections_all += layer.in_features * layer.out_features
    h = layer(h)

# Now, trace per-block metrics
print(f"{'Block':12s} | {'Parameters':>15s} ({'%':>5s}) | {'Connections (MACs)':>20s} ({'%':>5s})")
print("-" * 68)

h = torch.randn(1, 3, 224, 224)
block_params = 0
block_connections = 0
block_idx = 1

for layer in features:
    h = layer(h)
    p = sum(q.numel() for q in layer.parameters())
    block_params += p
    if isinstance(layer, nn.Conv2d):
        block_connections += h.shape[2] * h.shape[3] * layer.kernel_size[0] * layer.kernel_size[1] * layer.in_channels * layer.out_channels

    if isinstance(layer, nn.MaxPool2d):
        param_pct = (block_params / total_params_all) * 100
        conn_pct = (block_connections / total_connections_all) * 100
        print(f"Block {block_idx:<5d} | {block_params:>15,} ({param_pct:5.2f}%) | {block_connections:>20,} ({conn_pct:5.2f}%)")
        block_params = 0
        block_connections = 0
        block_idx += 1

# FC Block (Classifier)
fc_params = sum(p.numel() for layer in classifier for p in layer.parameters())
fc_connections = 0
for layer in classifier:
    if isinstance(layer, nn.Linear):
        fc_connections += layer.in_features * layer.out_features

fc_param_pct = (fc_params / total_params_all) * 100
fc_conn_pct = (fc_connections / total_connections_all) * 100
print(f"FC Classifier | {fc_params:>15,} ({fc_param_pct:5.2f}%) | {fc_connections:>20,} ({fc_conn_pct:5.2f}%)")
print("-" * 68)
print(f"{'Total':12s} | {total_params_all:>15,} (100.00%) | {total_connections_all:>20,} (100.00%)")

Block        |      Parameters (    %) |   Connections (MACs) (    %)
--------------------------------------------------------------------
Block 1     |          38,720 ( 0.03%) |        1,936,392,192 (12.52%)
Block 2     |         221,440 ( 0.16%) |        2,774,532,096 (17.93%)
Block 3     |       1,475,328 ( 1.07%) |        4,624,220,160 (29.89%)
Block 4     |       5,899,776 ( 4.26%) |        4,624,220,160 (29.89%)
Block 5     |       7,079,424 ( 5.12%) |        1,387,266,048 ( 8.97%)
FC Classifier |     123,642,856 (89.36%) |          123,633,664 ( 0.80%)
--------------------------------------------------------------------
Total        |     138,357,544 (100.00%) |       15,470,264,320 (100.00%)


138,357,544 — inside the paper's rounding. And the split is the thing to remember: the convolutional
stack holds about a tenth of the weights and does over 99% of the arithmetic, while the
fully-connected layers hold nine tenths and do under one per cent.

A convolutional weight is reused at every spatial position; a fully-connected weight is used once.

---

## 4. What local response normalization costs

Table 2 gives A and A-LRN the same parameter count, which is correct: LRN has no parameters. The paper
says only that it "leads to increased memory consumption and computation time", with no figure.

This is a first probe on a cut-down stack, not the full configuration — the full one comes in §5.

In [16]:
import torch, torch.nn as nn, time

def make_A(with_lrn):
    layers = [nn.Conv2d(3, 64, 3, padding=1), nn.ReLU()]

    if with_lrn:
        layers += [nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0)]

    layers += [
        nn.MaxPool2d(2, 2),
        nn.Conv2d( 64, 128, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
        nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
        nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
        nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Linear(512 * 7 * 7, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 4096),        nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 1000),
    ]
    return nn.Sequential(*layers)


def benchmark(model, batch=32, warmup=5, runs=20):
    model = model.cuda()
    x = torch.randn(batch, 3, 224, 224, device="cuda")

    # 1. warmup -- pehle kuch chakkar phenk do
    for _ in range(warmup):
        model.zero_grad(set_to_none=True)
        model(x).sum().backward()
    torch.cuda.synchronize()

    # 2. memory naapo
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model.zero_grad(set_to_none=True)
    model(x).sum().backward()
    torch.cuda.synchronize()
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2

    # 3. time naapo
    times = []
    for _ in range(runs):
        model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x).sum().backward()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times.sort()
    median_ms = times[len(times) // 2] * 1000
    return median_ms, peak_mb


results = {}
for name, lrn in [("A", False), ("A-LRN", True)]:
    ms, mb = benchmark(make_A(lrn))
    results[name] = (ms, mb)
    print(f"{name:6s}  {ms:7.2f} ms/step   {mb:8.1f} MB peak")

t0, m0 = results["A"]
t1, m1 = results["A-LRN"]
print()
print(f"LRN ka kharcha:  time +{(t1/t0 - 1)*100:.1f}%    memory +{m1 - m0:.1f} MB (+{(m1/m0 - 1)*100:.1f}%)")

A         27.41 ms/step     2609.3 MB peak
A-LRN     46.15 ms/step     4604.8 MB peak

LRN ka kharcha:  time +68.3%    memory +1995.5 MB (+76.5%)


**+68.3% time and +1995.5 MB**, for a layer with no parameters and no arithmetic.

Note the memory figure. In §5 the same comparison on the *full* configurations gives +1995.3 MB — the
same delta from a much larger network, because what LRN costs depends on the tensor it normalises (the
first convolution's output, 392 MB at this batch) and not on anything after it.

---

## 5. All six configurations, with the checks written first

Table 1 in code. The important part is not the builder, it is the block of invariants above it: every
fact that must hold if the table is read correctly, asserted before anything is benchmarked.

This is what caught the bug. An earlier version of the builder sliced the configuration list from the
wrong index, which silently skipped one layer and added an extra pool. The network still ran, still
produced plausible numbers, and would have gone into the write-up unnoticed — it crashed nothing. The
invariant on the pool count is what stopped it.

In [27]:
"""
VGG (Simonyan & Zisserman, ICLR 2015) -- Table 1 configurations, measured.

Two kinds of numbers appear below, and they are never mixed:

  COMPUTED   parameters and MACs, derived from the config alone and
             cross-checked against a real nn.Module. Deterministic.
  MEASURED   step time and peak GPU memory, timed on this machine.
             These are numbers the paper never published.
  PAPER      accuracy figures, quoted from the paper. Not reproducible
             here -- the authors used 4 Titan Black GPUs for 2-3 weeks
             per net on ImageNet.
"""

import time
import torch
import torch.nn as nn

# --------------------------------------------------------------------------
# Table 1 of the paper, one list per column.
#
#   (channels, kernel)  a conv layer, always followed by ReLU
#   'M'                 max pool, 2x2, stride 2
#   'L'                 Local Response Normalisation (no params, no MACs)
#
# Every column is written out in full. A-LRN is NOT derived from A by
# slicing: an off-by-one in a slice silently deletes a layer, the network
# still runs, and the benchmark is quietly wrong.
# --------------------------------------------------------------------------
CFG = {
    'A':     [(64, 3), 'M',
              (128, 3), 'M',
              (256, 3), (256, 3), 'M',
              (512, 3), (512, 3), 'M',
              (512, 3), (512, 3), 'M'],

    'A-LRN': [(64, 3), 'L', 'M',
              (128, 3), 'M',
              (256, 3), (256, 3), 'M',
              (512, 3), (512, 3), 'M',
              (512, 3), (512, 3), 'M'],

    'B':     [(64, 3), (64, 3), 'M',
              (128, 3), (128, 3), 'M',
              (256, 3), (256, 3), 'M',
              (512, 3), (512, 3), 'M',
              (512, 3), (512, 3), 'M'],

    'C':     [(64, 3), (64, 3), 'M',
              (128, 3), (128, 3), 'M',
              (256, 3), (256, 3), (256, 1), 'M',
              (512, 3), (512, 3), (512, 1), 'M',
              (512, 3), (512, 3), (512, 1), 'M'],

    'D':     [(64, 3), (64, 3), 'M',
              (128, 3), (128, 3), 'M',
              (256, 3), (256, 3), (256, 3), 'M',
              (512, 3), (512, 3), (512, 3), 'M',
              (512, 3), (512, 3), (512, 3), 'M'],

    'E':     [(64, 3), (64, 3), 'M',
              (128, 3), (128, 3), 'M',
              (256, 3), (256, 3), (256, 3), (256, 3), 'M',
              (512, 3), (512, 3), (512, 3), (512, 3), 'M',
              (512, 3), (512, 3), (512, 3), (512, 3), 'M'],
}

ORDER = ['A', 'A-LRN', 'B', 'C', 'D', 'E']

# Table 2 of the paper: parameter counts in millions.
PAPER_PARAMS_M = {'A': 133, 'A-LRN': 133, 'B': 133, 'C': 134, 'D': 138, 'E': 144}

# Table 3 of the paper: single-scale top-1 / top-5 validation error at S=Q=256.
PAPER_ERROR = {'A': (29.6, 10.4), 'A-LRN': (29.7, 10.5), 'B': (28.7, 9.9),
               'C': (28.1, 9.4),  'D': (27.0, 8.8),      'E': (27.3, 9.0)}

INPUT_SIZE = 224
FC_WIDTH = 4096
NUM_CLASSES = 1000


def build(name):
    """Assemble one configuration as an nn.Sequential."""
    layers = []
    in_c = 3
    for item in CFG[name]:
        if item == 'M':
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        elif item == 'L':
            # Parameters taken from AlexNet, as the paper instructs.
            layers.append(nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0))
        else:
            out_c, k = item
            # padding = k // 2 keeps the spatial size unchanged for any odd k.
            layers.append(nn.Conv2d(in_c, out_c, kernel_size=k, padding=k // 2))
            layers.append(nn.ReLU(inplace=True))
            in_c = out_c

    layers += [
        nn.Flatten(),
        nn.Linear(512 * 7 * 7, FC_WIDTH), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        nn.Linear(FC_WIDTH, FC_WIDTH),    nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        nn.Linear(FC_WIDTH, NUM_CLASSES),
    ]
    return nn.Sequential(*layers)


def static_metrics(name):
    """Parameters and MACs from the config alone -- no tensors allocated.

    A MAC is one multiply-accumulate. Bias terms count as parameters but
    not as MACs, which is the usual convention.
    """
    size, in_c = INPUT_SIZE, 3
    conv_params = conv_macs = 0

    for item in CFG[name]:
        if item == 'M':
            size //= 2
        elif item == 'L':
            continue                      # LRN has no parameters and no MACs
        else:
            out_c, k = item
            conv_params += k * k * in_c * out_c + out_c
            conv_macs += k * k * in_c * out_c * size * size
            in_c = out_c

    flat = in_c * size * size             # 512 * 7 * 7 = 25088
    fc_params = (flat * FC_WIDTH + FC_WIDTH
                 + FC_WIDTH * FC_WIDTH + FC_WIDTH
                 + FC_WIDTH * NUM_CLASSES + NUM_CLASSES)
    fc_macs = flat * FC_WIDTH + FC_WIDTH * FC_WIDTH + FC_WIDTH * NUM_CLASSES

    return {'conv_params': conv_params, 'fc_params': fc_params,
            'conv_macs': conv_macs, 'fc_macs': fc_macs,
            'params': conv_params + fc_params, 'macs': conv_macs + fc_macs,
            'flat': flat,
            'weight_layers': sum(1 for i in CFG[name] if i not in ('M', 'L')) + 3}


def check_invariants(static):
    """Facts that must hold. If one fails, the config table is wrong."""
    checks = []

    ok = static['A']['params'] == static['A-LRN']['params']
    checks.append(('A-LRN has the same parameters as A (LRN has none)', ok))

    ok = static['A']['macs'] == static['A-LRN']['macs']
    checks.append(('A-LRN has the same MACs as A', ok))

    ok = all(s['flat'] == 25088 for s in static.values())
    checks.append(('every config ends at 512x7x7 = 25088', ok))

    ok = len({s['fc_params'] for s in static.values()}) == 1
    checks.append(('the classifier is identical in all six configs', ok))

    for name in ORDER:
        got = round(static[name]['params'] / 1e6)
        checks.append((f'{name}: {got}M matches Table 2 ({PAPER_PARAMS_M[name]}M)',
                       got == PAPER_PARAMS_M[name]))

    for name in ORDER:
        real = sum(p.numel() for p in build(name).parameters())
        checks.append((f'{name}: analytic count matches the real nn.Module',
                       real == static[name]['params']))
    return checks

# ==========================================================================
# MEASURED -- step time and peak memory. These require a GPU.
# ==========================================================================

def benchmark(model, device, batch=32, warmup=10, runs=25):
    """Median wall-clock time for one forward+backward step, and peak memory.

    Three things make GPU timing honest:
      warmup        the first steps pay one-off costs (CUDA context, cuDNN
                    algorithm selection) that are not part of steady state
      synchronize   the GPU runs asynchronously; without this you time how
                    long it takes to QUEUE the work, not to do it
      median        one unlucky step (OS scheduling) must not move the result
    """
    model = model.to(device)
    x = torch.randn(batch, 3, INPUT_SIZE, INPUT_SIZE, device=device)

    for _ in range(warmup):
        model.zero_grad(set_to_none=True)
        model(x).sum().backward()
    torch.cuda.synchronize()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model.zero_grad(set_to_none=True)
    model(x).sum().backward()
    torch.cuda.synchronize()
    peak_mb = torch.cuda.max_memory_allocated() / 1024 ** 2

    times = []
    for _ in range(runs):
        model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x).sum().backward()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times.sort()
    return times[len(times) // 2] * 1000, peak_mb


# What changes between consecutive configs -- used for the marginal analysis.
STEP_NOTE = {
    ('A', 'B'): 'second conv added to blocks 1 and 2   (224x224, 112x112)',
    ('B', 'C'): 'three 1x1 convs added                 (56, 28, 14)',
    ('C', 'D'): 'those same three become 3x3           (56, 28, 14)',
    ('D', 'E'): 'fourth conv added to blocks 3, 4, 5   (56, 28, 14)',
}


def report(static, measured, batch):
    bar = '=' * 86

    print('\n' + bar)
    print('COMPUTED -- parameters and arithmetic')
    print(bar)
    print(f"{'cfg':7s} {'layers':>7s} {'conv params':>14s} {'fc params':>14s} "
          f"{'total':>14s} {'paper':>7s} {'GMACs':>8s}")
    for n in ORDER:
        s = static[n]
        print(f"{n:7s} {s['weight_layers']:>7d} {s['conv_params']:>14,} "
              f"{s['fc_params']:>14,} {s['params']:>14,} "
              f"{PAPER_PARAMS_M[n]:>6d}M {s['macs']/1e9:>8.2f}")

    d = static['D']
    print(f"\n  Where the weight sits vs where the work happens (config D):")
    print(f"    conv : {d['conv_params']/d['params']*100:5.2f}% of parameters, "
          f"{d['conv_macs']/d['macs']*100:5.2f}% of MACs")
    print(f"    fc   : {d['fc_params']/d['params']*100:5.2f}% of parameters, "
          f"{d['fc_macs']/d['macs']*100:5.2f}% of MACs")
    print('    A conv weight is reused at every spatial position; an fc weight')
    print('    is used exactly once. That single fact inverts the two columns.')

    print(f"\n  Depth is cheap, width is not:")
    print(f"    A -> E  conv params  {static['A']['conv_params']:>12,} -> "
          f"{static['E']['conv_params']:>12,}  "
          f"({static['E']['conv_params']/static['A']['conv_params']:.2f}x)")
    print(f"    A -> E  total params {static['A']['params']:>12,} -> "
          f"{static['E']['params']:>12,}  "
          f"({static['E']['params']/static['A']['params']:.2f}x)")
    print('    The classifier is identical in all six, so it dilutes the growth.')

    if not measured:
        print('\n(no GPU available -- skipping measured section)')
        return

    print('\n' + bar)
    print(f'MEASURED -- forward+backward, batch {batch}, this machine')
    print(bar)
    base_ms = measured['A'][0]
    base_g = static['A']['macs'] / 1e9
    print(f"{'cfg':7s} {'ms/step':>9s} {'img/s':>8s} {'peak MB':>10s} "
          f"{'time vs A':>10s} {'MACs vs A':>10s}")
    for n in ORDER:
        ms, mb = measured[n]
        print(f"{n:7s} {ms:>9.2f} {batch/(ms/1000):>8.1f} {mb:>10.1f} "
              f"{ms/base_ms:>9.2f}x {static[n]['macs']/1e9/base_g:>9.2f}x")

    print('\n' + bar)
    print('ANALYSIS -- marginal cost of each architectural step')
    print(bar)
    print(f"{'step':10s} {'+GMACs':>8s} {'+ms':>8s} {'ms/GMAC':>9s}   what changed")
    chain = ['A', 'B', 'C', 'D', 'E']
    rates = {}
    for a, b in zip(chain, chain[1:]):
        dg = (static[b]['macs'] - static[a]['macs']) / 1e9
        dm = measured[b][0] - measured[a][0]
        rates[(a, b)] = dm / dg
        print(f"{a + '->' + b:10s} {dg:>8.2f} {dm:>8.2f} {dm/dg:>9.2f}   {STEP_NOTE[(a,b)]}")

    r_1x1, r_3x3 = rates[('B', 'C')], rates[('C', 'D')]
    print(f"\n  1x1 vs 3x3, same blocks, same channels, same resolution:")
    print(f"    adding 1x1 convs : {r_1x1:5.2f} ms per GMAC")
    print(f"    adding 3x3 convs : {r_3x3:5.2f} ms per GMAC")
    print(f"    the 1x1 work costs {r_1x1/r_3x3:.2f}x more per unit of compute.")
    print('    A 3x3 conv does 9 multiply-accumulates per weight it loads from')
    print('    memory; a 1x1 does 1. Same traffic, one ninth of the work.')

    r_early, r_late = rates[('A', 'B')], rates[('C', 'D')]
    print(f"\n  Early layers vs late layers:")
    print(f"    MACs added at 224/112 : {r_early:5.2f} ms per GMAC")
    print(f"    MACs added at 56/28/14: {r_late:5.2f} ms per GMAC")
    print(f"    the same GMAC costs {r_early/r_late:.2f}x more early in the network.")

    a_ms, a_mb = measured['A']
    l_ms, l_mb = measured['A-LRN']
    print(f"\n  Local Response Normalisation (A-LRN vs A):")
    print(f"    time   {a_ms:7.2f} -> {l_ms:7.2f} ms   (+{(l_ms/a_ms-1)*100:.1f}%)")
    print(f"    memory {a_mb:7.1f} -> {l_mb:7.1f} MB   (+{l_mb-a_mb:.1f} MB)")
    print('    Identical parameters and MACs, so none of this shows up in')
    print('    Table 2. PyTorch does not fuse LRN: it is ~10 separate ops, each')
    print('    allocating a full-size tensor. Caffe, which the authors used, had')
    print('    a fused kernel -- so this is OUR cost, not theirs.')

    print('\n' + bar)
    print('PAPER -- quoted from Table 3, not reproducible here')
    print(bar)
    print(f"{'cfg':7s} {'top-1 err':>10s} {'top-5 err':>10s}")
    for n in ORDER:
        t1, t5 = PAPER_ERROR[n]
        print(f"{n:7s} {t1:>10.1f} {t5:>10.1f}")
    print('\n  A-LRN is worse than A, so B-E drop normalisation entirely.')
    print('  D beats C: both are 16 layers, but 3x3 captures spatial context')
    print('  that 1x1 cannot. The measured section shows D is also the cheaper')
    print('  of the two per unit of compute -- a second reason the paper never')
    print('  had. E is worse than D here: depth stops paying at 19 layers.')


if __name__ == '__main__':
    BATCH = 32
    use_gpu = torch.cuda.is_available()
    device = 'cuda' if use_gpu else 'cpu'
    if use_gpu:
        torch.backends.cudnn.benchmark = True   # autotune conv algorithms

    static = {n: static_metrics(n) for n in ORDER}

    print('=' * 86)
    print('INVARIANT CHECKS -- facts that must hold if the config table is right')
    print('=' * 86)
    failures = 0
    for text, ok in check_invariants(static):
        print(f'  [{"PASS" if ok else "FAIL"}]  {text}')
        failures += (not ok)
    if failures:
        raise SystemExit(f'\n{failures} invariant(s) failed -- fix the config before measuring.')
    print(f'\n  all checks passed')

    measured = {}
    if use_gpu:
        print(f'\nBenchmarking on {torch.cuda.get_device_name(0)} ...')
        for n in ORDER:
            model = build(n)
            measured[n] = benchmark(model, device, batch=BATCH)
            del model
            torch.cuda.empty_cache()
            print(f'  {n:7s} done')

    report(static, measured, BATCH)

INVARIANT CHECKS -- facts that must hold if the config table is right
  [PASS]  A-LRN has the same parameters as A (LRN has none)
  [PASS]  A-LRN has the same MACs as A
  [PASS]  every config ends at 512x7x7 = 25088
  [PASS]  the classifier is identical in all six configs
  [PASS]  A: 133M matches Table 2 (133M)
  [PASS]  A-LRN: 133M matches Table 2 (133M)
  [PASS]  B: 133M matches Table 2 (133M)
  [PASS]  C: 134M matches Table 2 (134M)
  [PASS]  D: 138M matches Table 2 (138M)
  [PASS]  E: 144M matches Table 2 (144M)
  [PASS]  A: analytic count matches the real nn.Module
  [PASS]  A-LRN: analytic count matches the real nn.Module
  [PASS]  B: analytic count matches the real nn.Module
  [PASS]  C: analytic count matches the real nn.Module
  [PASS]  D: analytic count matches the real nn.Module
  [PASS]  E: analytic count matches the real nn.Module

  all checks passed

Benchmarking on NVIDIA A100-SXM4-80GB ...
  A       done
  A-LRN   done
  B       done
  C       done
  D       done
  E 

All six configurations match Table 2 exactly, and the benchmark is the full-network version of §4:
**27.45 → 46.02 ms (+67.7%)** and **4624.1 → 6619.4 MB (+1995.3 MB)**.

The `ms/GMAC` block is a second finding, not in the paper. Adding 1×1 convolutions costs **4.77 ms per
GMAC**; adding 3×3 convolutions costs **1.66**. A 3×3 kernel does nine multiply-accumulates for every
weight it loads and a 1×1 does one. And the same arithmetic added early (224², 112²) costs 4.12 against
1.66 added late. GMAC count alone does not predict time.

---

## 6. Does it learn at all?

Before any accuracy number means anything, the question is whether a given initialisation lets the
network learn *anything*. The reference point is the loss of a network that has learned nothing:
uniform over 1,000 classes is exactly ln(1000) = 6.9078.

In [28]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def init_weights(model, mode):
    """Initialise every conv and linear layer with one scheme.

    paper_literal : the paper says "10^-2 variance", so std = sqrt(0.01) = 0.1
    paper_common  : how most implementations read that line, std = 0.01
    glorot        : Glorot & Bengio 2010 -- the fix the paper added after submission
    kaiming       : PyTorch's modern default for ReLU networks
    """
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if mode == 'paper_literal':
                nn.init.normal_(m.weight, mean=0.0, std=0.1)
            elif mode == 'paper_common':
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
            elif mode == 'glorot':
                nn.init.xavier_normal_(m.weight)
            elif mode == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)          # paper: "biases were initialised with zero"
    return model


def rms(t):
    """Root-mean-square. Comparable across layers of different sizes,
    unlike a raw norm, which just grows with the number of elements."""
    return t.detach().pow(2).mean().sqrt().item()


def profile(cfg, mode, batch=16, device='cuda'):
    """One forward and one backward. No training, no dataset."""
    torch.manual_seed(0)
    model = init_weights(build(cfg), mode).to(device)

    x = torch.randn(batch, 3, 224, 224, device=device)
    y = torch.randint(0, 1000, (batch,), device=device)

    activations, h = [], x
    for layer in model:
        h = layer(h)
        if isinstance(layer, nn.Conv2d):
            activations.append(rms(h))

    loss = F.cross_entropy(h, y)
    loss.backward()

    grads = [rms(m.weight.grad) for m in model.modules() if isinstance(m, nn.Conv2d)]
    return activations, grads, loss.item()


print('A network that has learned nothing guesses uniformly over 1000 classes.')
print(f'Its loss is exactly ln(1000) = {math.log(1000):.4f}. Remember that number.\n')

print(f"{'init':15s} {'loss':>16s} {'act first':>12s} {'act last':>12s} {'grad first':>12s} {'grad last':>12s}")
for mode in ['paper_literal', 'paper_common', 'glorot', 'kaiming']:
    a, g, l = profile('E', mode)
    print(f'{mode:15s} {l:>16.4f} {a[0]:>12.3e} {a[-1]:>12.3e} {g[0]:>12.3e} {g[-1]:>12.3e}')

print('\nPer-layer decay for the init that kills the network:')
a, g, l = profile('E', 'paper_common')
print(f"{'conv':>5s} {'activation RMS':>16s} {'grad RMS':>14s}")
for i, (ai, gi) in enumerate(zip(a, g), 1):
    print(f'{i:>5d} {ai:>16.3e} {gi:>14.3e}')

A network that has learned nothing guesses uniformly over 1000 classes.
Its loss is exactly ln(1000) = 6.9078. Remember that number.

init                        loss    act first     act last   grad first    grad last
paper_literal   247010230272.0000    5.098e-01    1.337e+08    5.166e+09    1.823e+08
paper_common              6.9078    5.098e-02    1.337e-08    3.364e-09    1.061e-10
glorot                    6.9086    2.936e-01    1.792e-03    2.006e-04    2.477e-05
kaiming                  23.3518    1.388e+00    2.816e+00    1.533e-01    7.049e-02

Per-layer decay for the init that kills the network:
 conv   activation RMS       grad RMS
    1        5.098e-02      3.364e-09
    2        8.326e-03      6.911e-10
    3        2.342e-03      5.613e-10
    4        5.490e-04      3.835e-10
    5        1.658e-04      3.215e-10
    6        5.096e-05      2.235e-10
    7        1.578e-05      1.973e-10
    8        5.467e-06      1.786e-10
    9        2.146e-06      1.472e-10
   10 

Four initialisations, and they fail in four different ways.

`paper_literal` (std 0.1) does not fail quietly — loss **2.47e+11**, activations growing to 1.3e+08.
`paper_common` (std 0.01) sits exactly at 6.9078: the network is not broken, it is *silent*. The
per-layer table shows why — activation RMS falls from 5.1e-02 to 2.4e-07 over twelve layers, and the
gradient with it. There is nothing left at the far end to learn from.

`kaiming` starts at 23.35, which looks worse and is not: it is the only one with signal to work with.

The gradient profile says which initialisations are broken. It does not say whether the survivors can
train. For that: can the network overfit a single batch? If it cannot memorise sixteen images, nothing
about a real dataset will help it.

In [29]:
def overfit_one_batch(cfg, mode, steps=150, batch=16, lr=1e-4, device='cuda'):
    """Show the same 16 images over and over. A healthy network memorises
    them and the loss collapses. A dead network cannot, and sits at ln(1000)."""
    torch.manual_seed(0)
    model = init_weights(build(cfg), mode).to(device).train()

    x = torch.randn(batch, 3, 224, 224, device=device)
    y = torch.randint(0, 1000, (batch,), device=device)

    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    history = []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        history.append(loss.item())
    return history


UNIFORM = math.log(1000)
print(f'A network that learns nothing stays at ln(1000) = {UNIFORM:.4f}\n')
print(f"{'init':15s} {'step 0':>12s} {'step 50':>12s} {'step 149':>12s} {'best':>12s}   verdict")

for mode in ['paper_literal', 'paper_common', 'glorot', 'kaiming']:
    h = overfit_one_batch('E', mode)
    best = min(h)
    learned = best < UNIFORM - 0.1
    print(f'{mode:15s} {h[0]:>12.4f} {h[50]:>12.4f} {h[149]:>12.4f} {best:>12.4f}   '
          f'{"learned something" if learned else "LEARNED NOTHING"}')

A network that learns nothing stays at ln(1000) = 6.9078

init                  step 0      step 50     step 149         best   verdict
paper_literal   247010230272.0000          nan          nan 247010230272.0000   LEARNED NOTHING
paper_common          6.9078       6.9047       6.8970       6.8970   LEARNED NOTHING
glorot                6.9086       6.9018       6.8872       6.8872   LEARNED NOTHING
kaiming              23.3518       3.3903       1.2505       1.2505   learned something


Only Kaiming moves. The other three sit at ln(1000) for 150 steps on sixteen images.

One learning rate is not a verdict on an initialisation — it is a verdict on that pair. The grid
separates the two.

In [30]:
LRS = [1e-4, 1e-3, 1e-2, 1e-1]
UNIFORM = math.log(1000)

print(f'ln(1000) = {UNIFORM:.4f}   |  "learned" = best loss fell more than 0.1 below it\n')
print(f"{'init':15s} {'lr':>8s} {'step 0':>10s} {'step 149':>10s} {'best':>10s}   verdict")

for mode in ['paper_common', 'glorot', 'kaiming']:
    for lr in LRS:
        h = overfit_one_batch('E', mode, steps=150, batch=16, lr=lr)
        best = min(h)
        if math.isnan(best) or math.isnan(h[-1]):
            verdict = 'diverged to NaN'
        elif best < UNIFORM - 0.1:
            verdict = 'LEARNED'
        else:
            verdict = 'nothing'
        print(f'{mode:15s} {lr:>8.0e} {h[0]:>10.4f} {h[-1]:>10.4f} {best:>10.4f}   {verdict}')
    print()

ln(1000) = 6.9078   |  "learned" = best loss fell more than 0.1 below it

init                  lr     step 0   step 149       best   verdict
paper_common       1e-04     6.9078     6.8970     6.8970   nothing
paper_common       1e-03     6.9078     6.7978     6.7978   LEARNED
paper_common       1e-02     6.9078     2.7827     2.5746   LEARNED
paper_common       1e-01     6.9078        nan     2.8416   diverged to NaN

glorot             1e-04     6.9086     6.8872     6.8872   nothing
glorot             1e-03     6.9086     2.8348     2.8348   LEARNED
glorot             1e-02     6.9086     2.5597     2.5597   LEARNED
glorot             1e-01     6.9086        nan     3.9027   diverged to NaN

kaiming            1e-04    23.3518     1.2174     1.2174   LEARNED
kaiming            1e-03    23.3518     2.6052     2.5257   LEARNED
kaiming            1e-02    23.3518        nan    23.3518   diverged to NaN
kaiming            1e-01    23.3518        nan    23.3518   diverged to NaN



And here is where one reading would have been wrong. `paper_common` looked dead in the profile — but at
lr=1e-2 it reaches **2.5746**, which is learning. The gradient was tiny, not absent, and a large enough
step makes use of it.

A one-shot gradient measurement predicts how a network starts, not whether it can train. The grid is
the measurement; the profile was a hypothesis.

---

## 7. Initialisation on real data

The single-batch test is a toy. Imagenette is 10 classes and ~9,500 training images — still not ILSVRC,
but a real dataset with real validation accuracy, and five epochs of it.

Both A (11 layers) and E (19 layers) are run, because the paper's claim is specifically about what
happens as depth grows.

In [32]:
"""
Does initialisation decide whether a deep VGG trains?

The paper (Simonyan & Zisserman 2015, Sect. 3.1) says deep configs could not
be trained from random init, so they trained the 11-layer A first and used its
weights to start the deeper nets. After submission they added that Glorot init
removes the need for that.

This tests the claim where it is testable: a real dataset, real epochs, a
shallow config and a deep one, several inits, each at more than one learning
rate -- because a learning rate tuned for one init is not neutral for another.

What this still is NOT: ImageNet. 10 classes, ~9.5k images. A negative result
here does not refute the paper; a positive one does not confirm it.
"""

import math, time, argparse
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

CFG = {
    'A': [(64,3),'M', (128,3),'M', (256,3),(256,3),'M',
          (512,3),(512,3),'M', (512,3),(512,3),'M'],
    'E': [(64,3),(64,3),'M', (128,3),(128,3),'M',
          (256,3),(256,3),(256,3),(256,3),'M',
          (512,3),(512,3),(512,3),(512,3),'M',
          (512,3),(512,3),(512,3),(512,3),'M'],
}

def build(name, num_classes=10):
    layers, in_c = [], 3
    for item in CFG[name]:
        if item == 'M':
            layers.append(nn.MaxPool2d(2, 2))
        else:
            out_c, k = item
            layers.append(nn.Conv2d(in_c, out_c, k, padding=k // 2))
            layers.append(nn.ReLU(inplace=True))
            in_c = out_c
    layers += [
        nn.Flatten(),
        nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(4096, 4096),        nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(4096, num_classes),
    ]
    return nn.Sequential(*layers)

def init_weights(model, mode):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if   mode == 'paper_literal': nn.init.normal_(m.weight, 0.0, 0.1)
            elif mode == 'paper_common':  nn.init.normal_(m.weight, 0.0, 0.01)
            elif mode == 'glorot':        nn.init.xavier_normal_(m.weight)
            elif mode == 'kaiming':       nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            else: raise ValueError(mode)
            nn.init.zeros_(m.bias)
    return model

# Paper, Sect. 2.1: "The only pre-processing we do is subtracting the mean RGB
# value, computed on the training set, from each pixel." No division by std --
# that is modern practice, not this paper's.
MEAN = [0.485, 0.456, 0.406]

def loaders(root, batch, workers=4):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.3, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=[1.0, 1.0, 1.0]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=[1.0, 1.0, 1.0]),
    ])
    tr = datasets.Imagenette(root, split='train', size='320px', download=True,  transform=train_tf)
    va = datasets.Imagenette(root, split='val',   size='320px', download=False, transform=val_tf)
    return (DataLoader(tr, batch, shuffle=True,  num_workers=workers, pin_memory=True, drop_last=True),
            DataLoader(va, batch, shuffle=False, num_workers=workers, pin_memory=True))

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    return 100.0 * correct / total

def train_one(cfg, mode, lr, tr, va, epochs, device, wd=5e-4):
    torch.manual_seed(0)
    model = init_weights(build(cfg), mode).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    history = []
    for ep in range(epochs):
        model.train()
        run, n = 0.0, 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            run += loss.item() * y.numel(); n += y.numel()
        history.append((run / n, evaluate(model, va, device)))
    return history


CHANCE = 10.0          # 10 classes, so a coin-flip model sits here
UNIFORM_LOSS = math.log(10)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--root', default='./data')
    ap.add_argument('--epochs', type=int, default=5)
    ap.add_argument('--batch', type=int, default=32)
    ap.add_argument('--workers', type=int, default=4)
    ap.add_argument('--smoke', action='store_true',
                    help='run on synthetic data to check the loop, no download')
    args, _ = ap.parse_known_args()   # parse_known_args: ignore Jupyter/Colab's own -f kernel.json

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        torch.backends.cudnn.benchmark = True

    if args.smoke:
        from torch.utils.data import TensorDataset
        x = torch.randn(64, 3, 224, 224); y = torch.randint(0, 10, (64,))
        tr = DataLoader(TensorDataset(x, y), args.batch, shuffle=True)
        va = DataLoader(TensorDataset(x, y), args.batch)
        configs, modes, lrs, epochs = ['A'], ['kaiming'], [1e-3], 1
    else:
        tr, va = loaders(args.root, args.batch, args.workers)
        configs = ['A', 'E']
        modes = ['paper_common', 'glorot', 'kaiming']
        lrs = [1e-3, 1e-2]
        epochs = args.epochs
        print(f'train batches {len(tr)}  val batches {len(va)}  device {device}')

    print(f'\nchance accuracy for 10 classes = {CHANCE:.1f}%'
          f'   |  loss of a net that learned nothing = ln(10) = {UNIFORM_LOSS:.4f}\n')

    rows = []
    for cfg in configs:
        for mode in modes:
            for lr in lrs:
                t0 = time.time()
                hist = train_one(cfg, mode, lr, tr, va, epochs, device)
                losses = [h[0] for h in hist]
                accs = [h[1] for h in hist]
                dead = any(math.isnan(l) for l in losses)
                rows.append((cfg, mode, lr, losses, accs, dead, time.time() - t0))
                tag = 'NaN' if dead else f'{accs[-1]:.1f}%'
                print(f'  {cfg}  {mode:13s} lr={lr:<7.0e} -> val {tag:>7s}'
                      f'   ({time.time()-t0:.0f}s)')

    print('\n' + '=' * 94)
    print('RESULT -- final validation accuracy after training')
    print('=' * 94)
    print(f"{'cfg':4s} {'init':14s} {'lr':>8s} {'loss ep1':>9s} {'loss last':>10s} "
          f"{'val acc':>8s} {'vs chance':>10s}  verdict")
    for cfg, mode, lr, losses, accs, dead, secs in rows:
        if dead:
            print(f'{cfg:4s} {mode:14s} {lr:>8.0e} {"nan":>9s} {"nan":>10s} '
                  f'{"-":>8s} {"-":>10s}  diverged')
            continue
        gain = accs[-1] - CHANCE
        verdict = 'LEARNED' if gain > 5 else ('weak' if gain > 1 else 'nothing')
        print(f'{cfg:4s} {mode:14s} {lr:>8.0e} {losses[0]:>9.4f} {losses[-1]:>10.4f} '
              f'{accs[-1]:>7.1f}% {gain:>+9.1f}  {verdict}')

    print('\nRead it this way: the question is not which row is best. It is whether')
    print('the DEEP config (E) fails where the SHALLOW one (A) succeeds, under the')
    print('same init and the same learning rate. That comparison is the paper\'s claim.')


if __name__ == '__main__':
    main()

100%|██████████| 342M/342M [00:13<00:00, 26.2MB/s]


train batches 295  val batches 123  device cuda

chance accuracy for 10 classes = 10.0%   |  loss of a net that learned nothing = ln(10) = 2.3026

  A  paper_common  lr=1e-03   -> val    9.1%   (76s)
  A  paper_common  lr=1e-02   -> val    9.1%   (75s)
  A  glorot        lr=1e-03   -> val   31.5%   (75s)
  A  glorot        lr=1e-02   -> val   48.9%   (75s)
  A  kaiming       lr=1e-03   -> val   56.1%   (75s)
  A  kaiming       lr=1e-02   -> val   61.8%   (75s)
  E  paper_common  lr=1e-03   -> val    9.1%   (123s)
  E  paper_common  lr=1e-02   -> val    9.1%   (123s)
  E  glorot        lr=1e-03   -> val    9.1%   (123s)
  E  glorot        lr=1e-02   -> val    9.9%   (123s)
  E  kaiming       lr=1e-03   -> val   50.1%   (123s)
  E  kaiming       lr=1e-02   -> val    9.9%   (123s)

RESULT -- final validation accuracy after training
cfg  init                 lr  loss ep1  loss last  val acc  vs chance  verdict
A    paper_common      1e-03    2.3026     2.3021     9.1%      -0.9  nothing
A 

Depth is the whole story in this table. On **A**, Glorot reaches 48.9% and Kaiming 61.8% — both train.
On **E**, the same Glorot collapses to 9.1%, which is chance, while Kaiming still reaches 50.1%.

Nothing changed but eight more layers.

Two learning rates leave gaps. Configuration E is the one that fails, so the gaps get filled on E.

In [33]:
# init_study.py ke functions pehle se defined hain is session mein.
# Agar nahi, toh pehle woh cell dobara chala lo.

tr, va = loaders('./data', batch=32, workers=4)   # dataset already downloaded
device = 'cuda'

# Jo pehle se pata hai, taaki tasveer poori dikhe:
KNOWN = {
    ('paper_common', 1e-3): 9.1, ('paper_common', 1e-2): 9.1,
    ('glorot',       1e-3): 9.1, ('glorot',       1e-2): 9.9,
    ('kaiming',      1e-3): 50.1, ('kaiming',     1e-2): 9.9,
}
NEW_LRS = [3e-3, 3e-2, 1e-1]

print('config E only. Filling the gaps between the learning rates we already tried.\n')
results = dict(KNOWN)

for mode in ['paper_common', 'glorot', 'kaiming']:
    for lr in NEW_LRS:
        hist = train_one('E', mode, lr, tr, va, epochs=5, device=device)
        losses = [h[0] for h in hist]
        acc = hist[-1][1]
        moved = abs(losses[0] - losses[-1])
        nan = any(math.isnan(l) for l in losses)
        results[(mode, lr)] = float('nan') if nan else acc
        tag = 'NaN' if nan else f'{acc:.1f}%'
        print(f'  E  {mode:13s} lr={lr:<7.0e} -> {tag:>7s}   loss moved {moved:.4f}')

print('\n' + '=' * 72)
print('CONFIG E -- the learning-rate window for each init')
print('=' * 72)
all_lrs = sorted(set([1e-3, 1e-2] + NEW_LRS))
print(f"{'init':15s}" + ''.join(f'{lr:>11.0e}' for lr in all_lrs))
for mode in ['paper_common', 'glorot', 'kaiming']:
    cells = []
    for lr in all_lrs:
        v = results.get((mode, lr))
        cells.append('      -    ' if v is None else
                     ('        NaN' if math.isnan(v) else f'{v:>10.1f}%'))
    print(f'{mode:15s}' + ''.join(cells))
print('\nchance = 10.0%')

config E only. Filling the gaps between the learning rates we already tried.

  E  paper_common  lr=3e-03   ->    9.1%   loss moved 0.0003
  E  paper_common  lr=3e-02   ->    9.9%   loss moved 0.0003
  E  paper_common  lr=1e-01   ->    9.9%   loss moved 0.0017
  E  glorot        lr=3e-03   ->    9.1%   loss moved 0.0003
  E  glorot        lr=3e-02   ->    9.9%   loss moved 0.0026
  E  glorot        lr=1e-01   ->    9.9%   loss moved 0.0060
  E  kaiming       lr=3e-03   ->   48.7%   loss moved 0.6192
  E  kaiming       lr=3e-02   ->     NaN   loss moved nan
  E  kaiming       lr=1e-01   ->     NaN   loss moved nan

CONFIG E -- the learning-rate window for each init
init                 1e-03      3e-03      1e-02      3e-02      1e-01
paper_common          9.1%       9.1%       9.1%       9.9%       9.9%
glorot                9.1%       9.1%       9.9%       9.9%       9.9%
kaiming              50.1%      48.7%       9.9%        NaN        NaN

chance = 10.0%


Filling the gaps gives configuration E's actual window: Kaiming trains at 1e-3 and 3e-3, reaches chance
at 1e-2, and diverges to NaN from 3e-2 up.

So the window has **two different edges**, failing in two different ways, and it is narrower than one
order of magnitude. Glorot and the paper's init have no window at all here — "loss moved 0.0003" over
five epochs.

---

## 8. The paper's transplant

> "we began with training the configuration A ... Then, when training deeper architectures, we
> initialised the first four convolutional layers and the last three fully-connected layers with the
> layers of net A."

The specification is **by layer**, not by position. A's first four convolutions are conv1_1, conv2_1,
conv3_1, conv3_2 — and in E those sit at different positions, because E has two convolutions in the
early blocks where A has one. Copying A's first four into E's first four positions copies the right
weights into the wrong shapes, so the layers are matched by name.

In [35]:
def vgg_names(cfg_name):
    """VGG's own naming: conv<block>_<index>.

    Position alone is not enough. A's second conv is conv2_1; E's second
    conv is conv1_2. Copying by position would pair layers whose shapes
    do not even match.
    """
    names, blk, idx = [], 1, 1
    for item in CFG[cfg_name]:
        if item == 'M':
            blk += 1
            idx = 1
        else:
            names.append(f'conv{blk}_{idx}')
            idx += 1
    return names


def convs(model):
    return [x for x in model if isinstance(x, nn.Conv2d)]


def linears(model):
    return [x for x in model if isinstance(x, nn.Linear)]


@torch.no_grad()
def transplant(donor, donor_cfg, recipient, recip_cfg, n_conv=4):
    """Paper Sect. 3.1: the first four conv layers and the last three FC
    layers come from the trained shallow net; the rest stays random.

    The asserts are the point. If a shape ever fails to match, the name
    mapping is wrong and the experiment is meaningless -- better to crash
    than to silently copy the wrong tensor.
    """
    d = dict(zip(vgg_names(donor_cfg), convs(donor)))
    r = dict(zip(vgg_names(recip_cfg), convs(recipient)))

    copied = []
    for name in vgg_names(donor_cfg)[:n_conv]:
        src, dst = d[name], r[name]
        assert src.weight.shape == dst.weight.shape, \
            f'{name}: donor {tuple(src.weight.shape)} vs recipient {tuple(dst.weight.shape)}'
        dst.weight.copy_(src.weight)
        dst.bias.copy_(src.bias)
        copied.append(name)

    for i, (src, dst) in enumerate(zip(linears(donor), linears(recipient))):
        assert src.weight.shape == dst.weight.shape
        dst.weight.copy_(src.weight)
        dst.bias.copy_(src.bias)
        copied.append(f'fc{6 + i}')

    return copied

In [36]:
import copy

def train_model(cfg, mode, lr, tr, va, epochs, device, model=None):
    """Same as train_one, but returns the trained model too."""
    if model is None:
        torch.manual_seed(0)
        model = init_weights(build(cfg), mode).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    hist = []
    for _ in range(epochs):
        model.train()
        run = n = 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step()
            run += loss.item() * y.numel(); n += y.numel()
        hist.append((run / n, evaluate(model, va, device)))
    return model, hist


# ---- Step 1: the donor ----
# NOT the paper's donor. Their init cannot train A at all, so this is Kaiming.
print('training donor A (kaiming, lr=1e-2) ...')
donor, dh = train_model('A', 'kaiming', 1e-2, tr, va, epochs=5, device=device)
print(f'  donor A val acc = {dh[-1][1]:.1f}%\n')

# ---- Step 2: transplant into E and train ----
BASELINE = {('paper_common', 1e-3): 9.1, ('paper_common', 1e-2): 9.1,
            ('glorot', 1e-3): 9.1, ('glorot', 1e-2): 9.9,
            ('kaiming', 1e-3): 50.1, ('kaiming', 1e-2): 9.9}

print(f"{'rest init':14s} {'lr':>8s} {'E alone':>9s} {'E + A':>9s} {'change':>9s}")
for rest in ['paper_common', 'glorot', 'kaiming']:
    for lr in [1e-3, 1e-2]:
        torch.manual_seed(0)
        e = init_weights(build('E'), rest).to(device)
        copied = transplant(donor, 'A', e, 'E')
        _, h = train_model('E', rest, lr, tr, va, epochs=5, device=device, model=e)
        got = h[-1][1]
        base = BASELINE[(rest, lr)]
        print(f'{rest:14s} {lr:>8.0e} {base:>8.1f}% {got:>8.1f}% {got-base:>+8.1f}')

print(f'\ncopied from A: {copied}')
print('chance = 10.0%   |   donor A itself =', f'{dh[-1][1]:.1f}%')

training donor A (kaiming, lr=1e-2) ...
  donor A val acc = 58.3%

rest init            lr   E alone     E + A    change
paper_common      1e-03      9.1%      9.9%     +0.8
paper_common      1e-02      9.1%     10.2%     +1.1
glorot            1e-03      9.1%     49.9%    +40.8
glorot            1e-02      9.9%     10.2%     +0.3
kaiming           1e-03     50.1%     59.0%     +8.9
kaiming           1e-02      9.9%     10.2%     +0.3

copied from A: ['conv1_1', 'conv2_1', 'conv3_1', 'conv3_2', 'fc6', 'fc7', 'fc8']
chance = 10.0%   |   donor A itself = 58.3%


The row that matters is **glorot, lr=1e-3: 9.1% → 49.9%**, a jump of 40.8 points. Same architecture,
same data, same optimiser; seven layers initialised from a trained A instead of from Glorot.

That is the paper's claim, reproduced.

Two things this does not show, and both need saying. The donor here is **Kaiming**-trained, not trained
with the paper's own initialisation — because that initialisation does not train A either, so the
paper's donor could not be produced. And the donor itself scored 58.3%; retrained, the same
configuration gave 61.8% and 63.6% elsewhere in this session, a spread of 5.3 points. Any transplant
result inside that spread is not a finding. 40.8 points is outside it.

One more reading of §3.1 to settle. The paper specifies "a zero-mean Gaussian distribution with the
1e-2 variance" — and variance 1e-2 is standard deviation 0.1, not 0.01. Read literally:

In [37]:
print('config E, paper init read literally (variance 1e-2 -> std 0.1)\n')
for lr in [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]:
    hist = train_one('E', 'paper_literal', lr, tr, va, epochs=5, device=device)
    losses = [h[0] for h in hist]
    acc = hist[-1][1]
    nan = any(math.isnan(l) for l in losses)
    print(f'  lr={lr:<7.0e} -> {"NaN" if nan else f"{acc:.1f}%":>7s}   '
          f'loss ep1 {losses[0]:.4g}')

config E, paper init read literally (variance 1e-2 -> std 0.1)

  lr=1e-03   ->     NaN   loss ep1 nan
  lr=3e-03   ->     NaN   loss ep1 nan
  lr=1e-02   ->     NaN   loss ep1 nan
  lr=3e-02   ->     NaN   loss ep1 nan
  lr=1e-01   ->     NaN   loss ep1 nan


NaN at every rate. Variance 1e-2 means std 0.1, and at that scale configuration E does not produce a
number at all — so the paper's wording, read literally, describes an initialisation that cannot be used.
Read as std 0.01 it at least runs, and that is almost certainly what was meant.

---

## 9. Section 3.2 — the fully-connected layers become convolutions

From training to testing. Everything above was about getting a network to learn; everything below is
about how to ask a trained one for an answer.

The weights here are torchvision's pretrained VGG-16, not ours — the question from here on is about the
evaluation procedure, and that can be asked of any competent configuration D.

In [1]:
import torch
import torchvision

torch.manual_seed(0)

net = torchvision.models.vgg16(weights="IMAGENET1K_V1").eval()

print(net.classifier)
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    feat = net.features(x)
    pooled = net.avgpool(feat)
print("features out:", tuple(feat.shape))
print("after avgpool:", tuple(pooled.shape))

for name, p in net.classifier.named_parameters():
    print(name, tuple(p.shape), p.numel())

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 225MB/s]


Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)
features out: (1, 512, 7, 7)
after avgpool: (1, 512, 7, 7)
0.weight (4096, 25088) 102760448
0.bias (4096,) 4096
3.weight (4096, 4096) 16777216
3.bias (4096,) 4096
6.weight (1000, 4096) 4096000
6.bias (1000,) 1000


Only one layer in VGG ever required a 224×224 input: the first fully-connected one, which demands
exactly 7·7·512 = 25,088 numbers. The convolutions never cared.

A `Linear(25088, 4096)` and a `Conv2d(512, 4096, kernel_size=7)` hold the same 102,760,448 weights in
the same order, so the conversion is a reshape. That claim is asserted rather than assumed, because
`.view()` would succeed and scramble the weights under any other flatten order.

In [2]:
import copy
import torch.nn as nn

def to_dense(vgg):
    """Replace VGG's Linear classifier with an equivalent all-convolutional head.
    The avgpool is deliberately dropped: it would destroy the spatial resolution
    that dense evaluation exists to produce."""
    fc0, fc3, fc6 = vgg.classifier[0], vgg.classifier[3], vgg.classifier[6]

    conv0 = nn.Conv2d(512,  4096, kernel_size=7)
    conv3 = nn.Conv2d(4096, 4096, kernel_size=1)
    conv6 = nn.Conv2d(4096, 1000, kernel_size=1)

    with torch.no_grad():
        conv0.weight.copy_(fc0.weight.view(4096, 512, 7, 7))
        conv0.bias.copy_(fc0.bias)
        conv3.weight.copy_(fc3.weight.view(4096, 4096, 1, 1))
        conv3.bias.copy_(fc3.bias)
        conv6.weight.copy_(fc6.weight.view(1000, 4096, 1, 1))
        conv6.bias.copy_(fc6.bias)

    head = nn.Sequential(conv0, nn.ReLU(inplace=True),
                         conv3, nn.ReLU(inplace=True),
                         conv6)
    return nn.Sequential(copy.deepcopy(vgg.features), head).eval()


dense = to_dense(net)

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    a = net(x)
    b = dense(x)

print("fc head  :", tuple(a.shape))
print("conv head:", tuple(b.shape))

b = b.flatten(1)
print("max abs difference:", (a - b).abs().max().item())
print("same parameter count:",
      sum(p.numel() for p in net.parameters()) == sum(p.numel() for p in dense.parameters()))

assert torch.allclose(a, b, atol=1e-4), "conversion is NOT equivalent"
print("equivalent")

fc head  : (2, 1000)
conv head: (2, 1000, 1, 1)
max abs difference: 1.430511474609375e-06
same parameter count: True
equivalent


**1.4e-06** — one or two units in the last place of float32. Exactly zero would be the suspicious
answer, since a matrix multiply and a convolution accumulate in different orders.

Once the classifier is a convolution it can slide, and a bigger input gives a map instead of one
answer. Predicted from the formula first, then measured.

In [3]:
def expected_map(h, w):
    """Score-map size for an h x w input.
    Five 2x2 maxpools floor-halve each side, which composes to floor(side / 32).
    The 7x7 conv then has stride 1 and no padding, so it removes 6 more."""
    return h // 32 - 6, w // 32 - 6


print(f"{'input':>12}   {'score map':>11}   {'predicted':>11}   {'positions':>9}")
for h, w in [(224, 224), (256, 256), (288, 288), (384, 384),
             (512, 512), (256, 341), (384, 512)]:
    x = torch.randn(1, 3, h, w)
    with torch.no_grad():
        out = dense(x)
    got, want = tuple(out.shape[2:]), expected_map(h, w)
    assert got == want, f"{h}x{w}: got {got}, predicted {want}"
    print(f"{h:>5} x {w:<4}   {got[0]:>4} x {got[1]:<4}   {want[0]:>4} x {want[1]:<4}   "
          f"{got[0] * got[1]:>9}")

       input     score map     predicted   positions
  224 x 224       1 x 1         1 x 1              1
  256 x 256       2 x 2         2 x 2              4
  288 x 288       3 x 3         3 x 3              9
  384 x 384       6 x 6         6 x 6             36
  512 x 512      10 x 10       10 x 10           100
  256 x 341       2 x 4         2 x 4              8
  384 x 512       6 x 10        6 x 10            60


Every size predicted correctly. At 224 the map is a single position, so dense evaluation there *is* the
original network; at 384 it is 6×6 = 36 positions.

The network's stride is 32, so each position is a 224×224 window offset 32 pixels from the last — dense
evaluation is multi-crop at every 32-pixel offset, for the price of one pass, because the convolutional
work underneath is shared.

---

## 10. Dense evaluation, first attempt

Imagenette again, this time for evaluation rather than training. Ten classes, 3,925 validation images.

These are real ImageNet classes and torchvision's weights know all 1,000 of them, so the logits are
kept at full 1,000-way rather than restricted to ten — restricting makes the task so easy that
differences between protocols vanish into noise.

In [4]:
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
!tar -xzf imagenette2.tgz

The class mapping is the one thing that would be silent if it were wrong, so it is checked against
torchvision's own category names rather than taken on trust. A mapping off by one would not crash; it
would show up as "that class is hard".

In [5]:
import os
from torchvision.models import VGG16_Weights

VAL = "imagenette2/val"
categories = VGG16_Weights.IMAGENET1K_V1.meta["categories"]

# Hypothesis: these are the ImageNet-1k indices of Imagenette's ten wnids.
# Verified below against torchvision's own category names - not taken on trust.
WNID_TO_IDX = {
    "n01440764":   0,   # tench
    "n02102040": 217,   # English springer
    "n02979186": 482,   # cassette player
    "n03000684": 491,   # chain saw
    "n03028079": 497,   # church
    "n03394916": 566,   # French horn
    "n03417042": 569,   # garbage truck
    "n03425413": 571,   # gas pump
    "n03445777": 574,   # golf ball
    "n03888257": 701,   # parachute
}

wnids = sorted(os.listdir(VAL))
assert wnids == sorted(WNID_TO_IDX), f"unexpected folders: {wnids}"

total = 0
for w in wnids:
    n = len(os.listdir(os.path.join(VAL, w)))
    total += n
    print(f"{w}  idx {WNID_TO_IDX[w]:>3}  {categories[WNID_TO_IDX[w]]:<18} {n:>4} images")
print(f"\n{total} validation images across {len(wnids)} classes")

n01440764  idx   0  tench               387 images
n02102040  idx 217  English springer    395 images
n02979186  idx 482  cassette player     357 images
n03000684  idx 491  chain saw           386 images
n03028079  idx 497  church              409 images
n03394916  idx 566  French horn         394 images
n03417042  idx 569  garbage truck       389 images
n03425413  idx 571  gas pump            419 images
n03445777  idx 574  golf ball           399 images
n03888257  idx 701  parachute           390 images

3925 validation images across 10 classes


All ten map to the name torchvision gives that index. 3,925 images.

In [6]:
import torch, os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
net = net.to(device).eval()


class Imagenette(Dataset):
    """Validation images labelled with their ImageNet-1k index, not 0-9."""
    def __init__(self, root, transform):
        self.items, self.transform = [], transform
        for wnid in sorted(os.listdir(root)):
            d = os.path.join(root, wnid)
            for f in sorted(os.listdir(d)):
                self.items.append((os.path.join(d, f), WNID_TO_IDX[wnid]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label


# torchvision's pretrained weights were trained with this normalisation.
NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

baseline_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), NORM])
loader = DataLoader(Imagenette(VAL, baseline_tf), batch_size=64,
                    num_workers=4, pin_memory=True)


@torch.no_grad()
def evaluate(model, loader):
    top1 = top5 = n = 0
    hits, counts = {}, {}
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        top = model(x).topk(5, dim=1).indices
        top1 += (top[:, 0] == y).sum().item()
        top5 += (top == y[:, None]).any(dim=1).sum().item()
        n += y.numel()
        for c in y.unique().tolist():
            m = y == c
            hits[c] = hits.get(c, 0) + (top[m][:, 0] == c).sum().item()
            counts[c] = counts.get(c, 0) + int(m.sum())
    per_class = {c: 100 * hits[c] / counts[c] for c in counts}
    return 100 * (1 - top1 / n), 100 * (1 - top5 / n), n, per_class


e1, e5, n, per_class = evaluate(net, loader)
print(f"single centre crop, Q=256 -> 224    "
      f"top-1 error {e1:5.2f}%    top-5 error {e5:5.2f}%    n={n}\n")
for c in sorted(per_class):
    flag = "  <-- CHECK MAPPING" if per_class[c] < 40 else ""
    print(f"  {categories[c]:<18} {per_class[c]:5.1f}% correct{flag}")

single centre crop, Q=256 -> 224    top-1 error 14.73%    top-5 error  2.88%    n=3925

  tench               94.8% correct
  English springer    92.7% correct
  cassette player     44.0% correct
  chain saw           85.8% correct
  church              73.8% correct
  French horn         89.6% correct
  garbage truck       92.5% correct
  gas pump            88.5% correct
  golf ball           94.5% correct
  parachute           93.1% correct


14.73% top-1 at full 1,000-way. Cassette player at 44.0% is not a mapping error — a wrong mapping would
read near 0% — it is ImageNet having `tape player`, `CD player` and `radio` as separate classes.

A sanity check on the weights themselves before comparing anything: torchvision publishes an accuracy
for these, measured under exactly this protocol.

In [7]:
import json
meta = VGG16_Weights.IMAGENET1K_V1.meta
print("recipe:", meta.get("recipe"))
print("ILSVRC val:", json.dumps(meta.get("_metrics"), indent=2))

recipe: https://github.com/pytorch/vision/tree/main/references/classification#alexnet-and-vgg
ILSVRC val: {
  "ImageNet-1K": {
    "acc@1": 71.592,
    "acc@5": 90.382
  }
}


28.408% top-1 reported, and the recipe is `RandomResizedCrop`, not the paper's scale jittering. So
these weights have no training scale S, which is worth remembering when Table 4 comes up.

Now the actual dense evaluation: whole uncropped image, score map averaged in logit space, soft-max
after, and the flipped image's posteriors averaged in.

In [8]:
dense = dense.to(device).eval()


@torch.no_grad()
def evaluate_dense(root, Q, flip=True):
    """Paper 3.2: rescale smallest side to Q, run the fully-convolutional net on the
    whole uncropped image, spatially average the class score map, then soft-max.
    Original and flipped posteriors are averaged - soft-max first, then average."""
    tf = T.Compose([T.Resize(Q), T.ToTensor(), NORM])
    loader = DataLoader(Imagenette(root, tf), batch_size=1, num_workers=4)

    top1 = top5 = n = 0
    shapes = set()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model_out = dense(x)
        shapes.add(tuple(out.shape[2:]))
        probs = out.mean(dim=(2, 3)).softmax(dim=1)
        if flip:
            flipped = dense(torch.flip(x, dims=[3])).mean(dim=(2, 3)).softmax(dim=1)
            probs = (probs + flipped) / 2
        top = probs.topk(5, dim=1).indices
        top1 += int((top[:, 0] == y).sum())
        top5 += int((top == y[:, None]).any(dim=1).sum())
        n += int(y.numel())
    return 100 * (1 - top1 / n), 100 * (1 - top5 / n), n, shapes


e1, e5, n, shapes = evaluate_dense(VAL, Q=256, flip=True)
print(f"dense, Q=256, with flip    top-1 error {e1:5.2f}%   top-5 error {e5:5.2f}%   n={n}")
print(f"distinct score-map sizes seen: {len(shapes)}")
print(f"examples: {sorted(shapes)[:8]}")

dense, Q=256, with flip    top-1 error 13.89%   top-5 error  2.11%   n=3925
distinct score-map sizes seen: 33
examples: [(2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9)]


13.89% against the crop baseline's 14.73%. 33 distinct score-map shapes, because real images are not
square.

---

## 11. The comparison was confounded

14.73% against 13.89% looks like dense evaluation winning. It is not a clean comparison: the baseline
had no flip averaging and the dense run did, so the gap mixes two changes.

Four cells of a 2×2 separate them.

In [9]:
import numpy as np

@torch.no_grad()
def run(protocol, Q, flip):
    """Per-image top-1 and top-5 correctness, kept as boolean arrays so that
    two protocols can later be compared image by image, not just in aggregate.
      crop  : resize smallest side to Q, centre-crop 224, FC network
      dense : resize smallest side to Q, whole uncropped image, conv network"""
    if protocol == "crop":
        tf, model, bs = T.Compose([T.Resize(Q), T.CenterCrop(224),
                                   T.ToTensor(), NORM]), net, 64
    else:
        tf, model, bs = T.Compose([T.Resize(Q), T.ToTensor(), NORM]), dense, 1

    loader = DataLoader(Imagenette(VAL, tf), batch_size=bs, num_workers=4)

    def posterior(t):
        o = model(t)
        if o.dim() == 4:                 # dense: average the score map in logit space
            o = o.mean(dim=(2, 3))
        return o.softmax(dim=1)          # then soft-max, per the paper's ordering

    c1, c5 = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        p = posterior(x)
        if flip:
            p = (p + posterior(torch.flip(x, dims=[3]))) / 2
        top = p.topk(5, dim=1).indices
        c1.append((top[:, 0] == y).cpu().numpy())
        c5.append((top == y[:, None]).any(dim=1).cpu().numpy())
    return np.concatenate(c1), np.concatenate(c5)


results = {}
for protocol in ("crop", "dense"):
    for flip in (False, True):
        results[(protocol, flip)] = run(protocol, Q=256, flip=flip)
        c1, c5 = results[(protocol, flip)]
        print(f"{protocol:<6} Q=256  flip={str(flip):<5}  "
              f"top-1 error {100*(1-c1.mean()):5.2f}%   "
              f"top-5 error {100*(1-c5.mean()):5.2f}%")

np.save("results_q256.npy", results, allow_pickle=True)

err = lambda k: 100 * (1 - results[k][0].mean())
print()
print(f"dense over crop, no flip : {err(('crop',  False)) - err(('dense', False)):+.2f} pp")
print(f"dense over crop, w/ flip : {err(('crop',  True )) - err(('dense', True )):+.2f} pp")
print(f"flip, under crop         : {err(('crop',  False)) - err(('crop',  True )):+.2f} pp")
print(f"flip, under dense        : {err(('dense', False)) - err(('dense', True )):+.2f} pp")

assert abs(err(("crop", False)) - 14.73) < 0.01, "harness disagrees with the baseline"
print("\nbaseline reproduced")

crop   Q=256  flip=False  top-1 error 14.73%   top-5 error  2.88%
crop   Q=256  flip=True   top-1 error 14.27%   top-5 error  2.42%
dense  Q=256  flip=False  top-1 error 14.88%   top-5 error  2.29%
dense  Q=256  flip=True   top-1 error 13.89%   top-5 error  2.11%

dense over crop, no flip : -0.15 pp
dense over crop, w/ flip : +0.38 pp
flip, under crop         : +0.46 pp
flip, under dense        : +0.99 pp

baseline reproduced


Now the confound is visible. Flip averaging alone is worth 0.46 points under crop. Dense **without**
flip is 14.88% — slightly *worse* than the 14.73% baseline.

So the apparent win was mostly the flip.

Aggregate error rates cannot say whether two protocols disagree on two hundred images or on none. A
paired test can: it throws away every image the two agree on, and asks whether the rest split like a
fair coin.

In [10]:
from scipy.stats import binomtest

def mcnemar(a, b, label):
    """Paired comparison on the same images.
    a_only: a correct and b wrong.   b_only: a wrong and b correct.
    Under the null the split of those disagreements is a fair coin."""
    a_only = int((a & ~b).sum())
    b_only = int((~a & b).sum())
    n = a_only + b_only
    p = binomtest(b_only, n, 0.5).pvalue if n else 1.0
    print(f"{label:<28} agree {int((a == b).sum()):>4}   "
          f"only-1st {a_only:>3}   only-2nd {b_only:>3}   p = {p:.4f}"
          f"{'   significant' if p < 0.05 else ''}")

top1 = lambda k: results[k][0]

mcnemar(top1(("crop",  False)), top1(("dense", False)), "crop vs dense, no flip")
mcnemar(top1(("crop",  True )), top1(("dense", True )), "crop vs dense, with flip")
mcnemar(top1(("crop",  False)), top1(("crop",  True )), "no flip vs flip, crop")
mcnemar(top1(("dense", False)), top1(("dense", True )), "no flip vs flip, dense")

crop vs dense, no flip       agree 3729   only-1st 101   only-2nd  95   p = 0.7211
crop vs dense, with flip     agree 3766   only-1st  72   only-2nd  87   p = 0.2668
no flip vs flip, crop        agree 3841   only-1st  33   only-2nd  51   p = 0.0630
no flip vs flip, dense       agree 3818   only-1st  34   only-2nd  73   p = 0.0002   significant


`p = 0.72`. The 0.15-point difference is noise.

But look at the agreement column: the two protocols disagree on **196 images** — dense gets 95 right
that crop misses, crop gets 101 right that dense misses. Two genuinely different predictors whose wins
cancel. That is exactly why the paper can call them *complementary*.

That was top-1 only. The top-5 arrays were saved and not used — and the larger differences turn out to
be there.

In [11]:
print("TOP-5\n")
top5 = lambda k: results[k][1]

mcnemar(top5(("crop",  False)), top5(("dense", False)), "crop vs dense, no flip")
mcnemar(top5(("crop",  True )), top5(("dense", True )), "crop vs dense, with flip")
mcnemar(top5(("crop",  False)), top5(("crop",  True )), "no flip vs flip, crop")
mcnemar(top5(("dense", False)), top5(("dense", True )), "no flip vs flip, dense")

TOP-5

crop vs dense, no flip       agree 3872   only-1st  15   only-2nd  38   p = 0.0022   significant
crop vs dense, with flip     agree 3879   only-1st  17   only-2nd  29   p = 0.1038
no flip vs flip, crop        agree 3891   only-1st   8   only-2nd  26   p = 0.0029   significant
no flip vs flip, dense       agree 3910   only-1st   4   only-2nd  11   p = 0.1185


On top-5 dense wins clearly: **15 against 38, p = 0.0022**. Same images, same protocols, opposite
answer from top-1 — because top-1 scores rank 2 and rank 900 identically.

---

## 12. Where dense evaluation actually acts

Top-1 says nothing, top-5 says dense is better. Both can be true if dense moves answers up the list
without moving them to the front. The rank of the correct class under each protocol settles it.

In [12]:
@torch.no_grad()
def run_ranks(protocol, Q, flip=False):
    """Rank of the correct class for every image, 1 = top.
    Needs the full 1000-way ordering, so topk(5) is not enough here."""
    if protocol == "crop":
        tf, model, bs = T.Compose([T.Resize(Q), T.CenterCrop(224),
                                   T.ToTensor(), NORM]), net, 64
    else:
        tf, model, bs = T.Compose([T.Resize(Q), T.ToTensor(), NORM]), dense, 1

    loader = DataLoader(Imagenette(VAL, tf), batch_size=bs, num_workers=4)

    def posterior(t):
        o = model(t)
        if o.dim() == 4:
            o = o.mean(dim=(2, 3))
        return o.softmax(dim=1)

    out = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        p = posterior(x)
        if flip:
            p = (p + posterior(torch.flip(x, dims=[3]))) / 2
        true = p.gather(1, y[:, None])
        out.append(((p > true).sum(dim=1) + 1).cpu().numpy())   # count of classes above it
    return np.concatenate(out)


ranks = {p: run_ranks(p, Q=256, flip=False) for p in ("crop", "dense")}


def bucket(r):
    b = np.full(r.shape, 3)      # 3 = rank 6+
    b[r <= 5] = 2                # 2 = rank 2-5
    b[r == 1] = 1                # 1 = rank 1
    return b

bc, bd = bucket(ranks["crop"]), bucket(ranks["dense"])
name = {1: "rank 1", 2: "rank 2-5", 3: "rank 6+"}

print("rows = crop, columns = dense\n")
print(f"{'':<10}" + "".join(f"{name[j]:>10}" for j in (1, 2, 3)))
for i in (1, 2, 3):
    print(f"{name[i]:<10}" + "".join(
        f"{int(((bc == i) & (bd == j)).sum()):>10}" for j in (1, 2, 3)))

entered = (bc == 3) & (bd <= 2)
left    = (bc <= 2) & (bd == 3)
print(f"\nentered top-5 under dense : {int(entered.sum())}")
print(f"     landed at rank 1     : {int((entered & (bd == 1)).sum())}")
print(f"     landed at rank 2-5   : {int((entered & (bd == 2)).sum())}")
print(f"left top-5 under dense    : {int(left.sum())}")

print(f"\nmedian rank   crop {np.median(ranks['crop']):.1f}   "
      f"dense {np.median(ranks['dense']):.1f}")
print(f"mean log-rank crop {np.log(ranks['crop']).mean():.4f}   "
      f"dense {np.log(ranks['dense']).mean():.4f}")

rows = crop, columns = dense

              rank 1  rank 2-5   rank 6+
rank 1          3246       101         0
rank 2-5          85       365        15
rank 6+           10        28        75

entered top-5 under dense : 38
     landed at rank 1     : 10
     landed at rank 2-5   : 28
left top-5 under dense    : 15

median rank   crop 1.0   dense 1.0
mean log-rank crop 0.1884   dense 0.1776


The empty cell is the finding. Of the 3,347 images the centre crop ranked first, **zero** fell below
rank five under dense. And of the 38 that entered the top five, 28 landed at ranks 2–5 rather than
rank 1.

Dense improves the candidate list and not the top slot. Averaging over windows dilutes a confident
right answer; it does not destroy one.

(A plainer explanation competes: images the centre crop ranks first are the easy ones, and easy images
survive any window. These runs do not separate the two.)

Mean log-rank favours dense. Whether that survives a signed test is a different question.

In [13]:
from scipy.stats import wilcoxon

d = np.log(ranks["dense"]) - np.log(ranks["crop"])
moved = d != 0
print(f"images whose rank changed: {int(moved.sum())}")
print(wilcoxon(d[moved]))
print(f"improved {int((d < 0).sum())}   worsened {int((d > 0).sum())}")

images whose rank changed: 441
WilcoxonResult(statistic=np.float64(43968.0), pvalue=np.float64(0.07303009076695269))
improved 236   worsened 205


`p = 0.073`. Mean log-rank favoured dense, but 236 improved against 205 worsened — nearly even. The
mean was pulled by magnitude, and the claim does not survive.

---

## 13. Multi-crop, and the paper's actual claim

§4.3: 50 crops per scale, a 5×5 grid with both flips. Table 5's claim is that multi-crop and dense are
complementary — that combining them beats each alone.

In [14]:
labels = np.array([lab for _, lab in Imagenette(VAL, baseline_tf).items])


@torch.no_grad()
def posteriors(protocol, Q, flip=False, grid=5):
    """Full (N, 1000) posterior matrix for one protocol, so that accuracy, ranks and
    protocol combinations all derive from one GPU pass instead of one pass each.
      crop      : centre 224 crop, FC network
      dense     : whole uncropped image, conv network, score map averaged
      multicrop : 5x5 grid of 224 crops with 2 flips = 50 crops, FC network"""
    tf = T.Compose([T.Resize(Q), T.CenterCrop(224), T.ToTensor(), NORM]) \
         if protocol == "crop" else \
         T.Compose([T.Resize(Q), T.ToTensor(), NORM])
    bs = 64 if protocol == "crop" else 1
    loader = DataLoader(Imagenette(VAL, tf), batch_size=bs, num_workers=4)

    out = []
    for x, _ in loader:
        x = x.to(device)
        if protocol == "multicrop":
            H, W = x.shape[2], x.shape[3]
            ys = torch.linspace(0, H - 224, grid).round().long().tolist()
            xs = torch.linspace(0, W - 224, grid).round().long().tolist()
            crops = torch.cat([x[:, :, i:i + 224, j:j + 224] for i in ys for j in xs])
            crops = torch.cat([crops, torch.flip(crops, dims=[3])])   # 50 crops
            p = net(crops).softmax(dim=1).mean(dim=0, keepdim=True)
        else:
            model = net if protocol == "crop" else dense
            def post(t):
                o = model(t)
                return (o.mean(dim=(2, 3)) if o.dim() == 4 else o).softmax(dim=1)
            p = post(x)
            if flip:
                p = (p + post(torch.flip(x, dims=[3]))) / 2
        out.append(p.cpu().numpy())
    return np.concatenate(out)


P = {
    "crop":      posteriors("crop",      256, flip=True),
    "dense":     posteriors("dense",     256, flip=True),
    "multicrop": posteriors("multicrop", 256),
}
P["multicrop & dense"] = (P["multicrop"] + P["dense"]) / 2

for name, p in P.items():
    order = np.argsort(-p, axis=1)
    e1 = 100 * (1 - (order[:, 0] == labels).mean())
    e5 = 100 * (1 - (order[:, :5] == labels[:, None]).any(axis=1).mean())
    print(f"{name:<18} top-1 error {e1:5.2f}%   top-5 error {e5:5.2f}%")

crop               top-1 error 14.27%   top-5 error  2.42%
dense              top-1 error 13.89%   top-5 error  2.11%
multicrop          top-1 error 14.01%   top-5 error  1.96%
multicrop & dense  top-1 error 13.78%   top-5 error  1.96%


Which comparisons get tested is decided **before** the numbers are looked at. Running every pair and
reporting the ones under 0.05 is a different and much weaker thing.

In [15]:
def correct(p, k=1):
    order = np.argsort(-p, axis=1)
    return (order[:, :k] == labels[:, None]).any(axis=1)


print("PRE-SPECIFIED - the paper's claim that combining beats each alone")
print("two tests on top-1, so the threshold is 0.05 / 2 = 0.025\n")
comb = correct(P["multicrop & dense"])
mcnemar(correct(P["dense"]),     comb, "dense vs combination")
mcnemar(correct(P["multicrop"]), comb, "multicrop vs combination")

print("\nEXPLORATORY - reported for the record, no conclusions drawn")
for a, b in [("crop", "dense"), ("crop", "multicrop"), ("dense", "multicrop"),
             ("crop", "multicrop & dense")]:
    for k in (1, 5):
        mcnemar(correct(P[a], k), correct(P[b], k), f"  top-{k}  {a} vs {b}")

PRE-SPECIFIED - the paper's claim that combining beats each alone
two tests on top-1, so the threshold is 0.05 / 2 = 0.025

dense vs combination         agree 3861   only-1st  30   only-2nd  34   p = 0.7080
multicrop vs combination     agree 3862   only-1st  27   only-2nd  36   p = 0.3135

EXPLORATORY - reported for the record, no conclusions drawn
  top-1  crop vs dense       agree 3766   only-1st  72   only-2nd  87   p = 0.2668
  top-5  crop vs dense       agree 3879   only-1st  17   only-2nd  29   p = 0.1038
  top-1  crop vs multicrop   agree 3791   only-1st  62   only-2nd  72   p = 0.4370
  top-5  crop vs multicrop   agree 3879   only-1st  14   only-2nd  32   p = 0.0114   significant
  top-1  dense vs multicrop  agree 3802   only-1st  64   only-2nd  59   p = 0.7185
  top-5  dense vs multicrop  agree 3885   only-1st  17   only-2nd  23   p = 0.4296
  top-1  crop vs multicrop & dense agree 3796   only-1st  55   only-2nd  74   p = 0.1127
  top-5  crop vs multicrop & dense agree 3881   

Both pre-specified tests fail. The combination is ahead in both (34 vs 30, 36 vs 27) but by less than
a coin toss would explain.

This is not evidence against the paper. It is 3,925 images not being enough to decide it.

---

## 14. Getting the paper's own data

Nothing above reached significance. The power calculation said why: about 64 discordant pairs where
roughly 130 were needed — the experiment was about half the size it needed to be for this one claim.

Imagenette is also the wrong regime. Its ten classes are easily separated; the paper sits at 24.8%
top-1 error, this sits at 13.8%. So: the real ILSVRC-2012 validation set, all 50,000 images.

In [17]:
!pip -q install huggingface_hub

from huggingface_hub import login, hf_hub_download
login()          # paste the token into the box this opens, not into the chat

from huggingface_hub import list_repo_files

files = list_repo_files("timm/imagenet-1k-wds", repo_type="dataset")
print(len(files), "files in the repo\n")

val = sorted(f for f in files if "valid" in f.lower())
print(len(val), "validation files")
for f in val[:4]:
    print("  ", f)
print("   ...")
for f in val[-2:]:
    print("  ", f)

print("\nnon-tar files (metadata, label maps):")
for f in sorted(files):
    if not f.endswith(".tar"):
        print("  ", f)

1091 files in the repo

64 validation files
   imagenet1k-validation-00.tar
   imagenet1k-validation-01.tar
   imagenet1k-validation-02.tar
   imagenet1k-validation-03.tar
   ...
   imagenet1k-validation-62.tar
   imagenet1k-validation-63.tar

non-tar files (metadata, label maps):
   .gitattributes
   README.md
   _info.json


In [18]:
from huggingface_hub import hf_hub_download
import tarfile, os, json

info = json.load(open(hf_hub_download(
    "timm/imagenet-1k-wds", "_info.json", repo_type="dataset")))

print("top-level keys:", list(info.keys()))
for k, v in info.items():
    desc = f"{type(v).__name__}, len {len(v)}" if isinstance(v, (list, dict)) else repr(v)[:90]
    print(f"  {k:<22} {desc}")

path = hf_hub_download(
    "timm/imagenet-1k-wds", "imagenet1k-validation-00.tar", repo_type="dataset")
print(f"\nshard 00: {os.path.getsize(path) / 1e6:.1f} MB")

with tarfile.open(path) as t:
    names = t.getnames()
    print(f"{len(names)} members\n")
    for n in names[:9]:
        print("  ", n)

    stem = names[0].rsplit(".", 1)[0]
    print(f"\nall files for sample '{stem}':")
    for n in names:
        if n.startswith(stem):
            d = t.extractfile(n).read()
            shown = repr(d[:200]) if n.endswith((".json", ".cls", ".txt")) else f"<{len(d)} bytes>"
            print(f"   {n:<44} {shown}")

_info.json:   0%|          | 0.00/73.4k [00:00<?, ?B/s]

top-level keys: ['name', 'splits']
  name                   'imagenet1k'
  splits                 dict, len 2


imagenet1k-validation-00.tar: reconstructing file:   0%|          |  0.00B /  108MB            

imagenet1k-validation-00.tar: downloading bytes:           |  0.00B            


shard 00: 108.0 MB
2343 members

   ILSVRC2012_val_00009670.cls
   ILSVRC2012_val_00009670.jpg
   ILSVRC2012_val_00009670.json
   ILSVRC2012_val_00036705.cls
   ILSVRC2012_val_00036705.jpg
   ILSVRC2012_val_00036705.json
   ILSVRC2012_val_00003545.cls
   ILSVRC2012_val_00003545.jpg
   ILSVRC2012_val_00003545.json

all files for sample 'ILSVRC2012_val_00009670':
   ILSVRC2012_val_00009670.cls                  b'26'
   ILSVRC2012_val_00009670.jpg                  <65661 bytes>
   ILSVRC2012_val_00009670.json                 b'{"label": 26, "width": 600, "height": 450, "filename": "n01630670/ILSVRC2012_val_00009670.JPEG"}'


The label convention is the thing that would ruin 50,000 images silently, so it is checked before the
7 GB download rather than after. ImageNet class index = position of the wnid in sorted order — checked
for monotonicity on one shard first, then across all 1,000 classes.

In [19]:
import tarfile, json
from torchvision.models import VGG16_Weights
categories = VGG16_Weights.IMAGENET1K_V1.meta["categories"]

print("splits:", {k: (list(v.keys()) if isinstance(v, dict) else v)
                  for k, v in info["splits"].items()})

pairs = {}
with tarfile.open(path) as t:
    for n in t.getnames():
        if n.endswith(".json"):
            j = json.loads(t.extractfile(n).read())
            wnid = j["filename"].split("/")[0]
            if wnid in pairs:
                assert pairs[wnid] == j["label"], f"{wnid} carries two different labels"
            pairs[wnid] = j["label"]

ordered = sorted(pairs)
seq = [pairs[w] for w in ordered]
monotonic = all(a < b for a, b in zip(seq, seq[1:]))

print(f"\n{len(pairs)} distinct wnids in shard 00")
print(f"labels increase with sorted wnid order: {monotonic}")
print("\nspot check against torchvision's own names:")
for w in ordered[:4] + ordered[-4:]:
    print(f"   {w}   label {pairs[w]:>3}   {categories[pairs[w]]}")

assert monotonic, "label order breaks the sorted-wnid convention - stop and investigate"
print("\nconvention holds - downloading the remaining shards")

from huggingface_hub import snapshot_download
local = snapshot_download("timm/imagenet-1k-wds", repo_type="dataset",
                          allow_patterns="imagenet1k-validation-*.tar")
print("shards at:", local)

splits: {'train': ['name', 'filenames', 'shard_lengths', 'num_samples'], 'validation': ['name', 'filenames', 'shard_lengths', 'num_samples']}

551 distinct wnids in shard 00
labels increase with sorted wnid order: True

spot check against torchvision's own names:
   n01491361   label   3   tiger shark
   n01494475   label   4   hammerhead
   n01518878   label   9   ostrich
   n01530575   label  10   brambling
   n12985857   label 991   coral fungus
   n13037406   label 993   gyromitra
   n13054560   label 997   bolete
   n13133613   label 998   ear

convention holds - downloading the remaining shards


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

shards at: /root/.cache/huggingface/hub/datasets--timm--imagenet-1k-wds/snapshots/cdf403ce12f01022a0c36e584e588c0b9cebc4af


Monotonic, and the spot checks line up — `n01491361 → 3 → tiger shark`. The convention holds.

In [20]:
import tarfile, glob, os, json

OUT = "imagenet_val"
shards = sorted(glob.glob(f"{local}/imagenet1k-validation-*.tar"))
print(len(shards), "shards")

wnid_of_label, n = {}, 0
for s in shards:
    with tarfile.open(s) as t:
        for name in t.getnames():
            if not name.endswith(".json"):
                continue
            stem = name[:-5]
            j = json.loads(t.extractfile(name).read())
            wnid, lab = j["filename"].split("/")[0], j["label"]
            if lab in wnid_of_label:
                assert wnid_of_label[lab] == wnid, f"label {lab} maps to two wnids"
            wnid_of_label[lab] = wnid

            d = os.path.join(OUT, wnid)
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, stem + ".jpg"), "wb") as f:
                f.write(t.extractfile(stem + ".jpg").read())
            n += 1

print(f"{n} images extracted into {OUT}/")
print(f"{len(wnid_of_label)} distinct classes")
print(f"expected from _info.json: {info['splits']['validation']['num_samples']}")

assert len(wnid_of_label) == 1000, "not all 1000 classes present"
assert [wnid_of_label[i] for i in range(1000)] == sorted(wnid_of_label.values()), \
    "label index is NOT the sorted position of the wnid"
print("\nlabel i == sorted position of wnid i, verified across all 1000 classes")

64 shards
50000 images extracted into imagenet_val/
1000 distinct classes
expected from _info.json: 50000

label i == sorted position of wnid i, verified across all 1000 classes


50,000 images, 1,000 classes, and the stronger check passes: label *i* is the *i*-th wnid in sorted
order, across all 1,000.

---

## 15. Precision, then the real run

Half precision would make this faster. Whether it is safe depends on whether its noise is smaller than
the effects being measured — which is a thing to measure, not assume.

In [21]:
import numpy as np, os, time

class ImageNetVal(Dataset):
    """Labelled by the sorted position of the wnid folder, which was verified
    above to equal the ImageNet-1k class index."""
    def __init__(self, root, transform):
        self.items, self.transform = [], transform
        for i, wnid in enumerate(sorted(os.listdir(root))):
            d = os.path.join(root, wnid)
            for f in sorted(os.listdir(d)):
                self.items.append((os.path.join(d, f), i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        p, y = self.items[i]
        return self.transform(Image.open(p).convert("RGB")), y


in_labels = np.array([y for _, y in ImageNetVal(OUT, None).items])
np.save("in_labels.npy", in_labels)
print(f"{len(in_labels)} images, {in_labels.max() + 1} classes")


@torch.no_grad()
def crop_posteriors(Q=256, half=False):
    tf = T.Compose([T.Resize(Q), T.CenterCrop(224), T.ToTensor(), NORM])
    loader = DataLoader(ImageNetVal(OUT, tf), batch_size=128,
                        num_workers=8, pin_memory=True)
    out, t0 = [], time.time()
    for i, (x, _) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=half):
            p = net(x).float().softmax(dim=1)
            p = (p + net(torch.flip(x, dims=[3])).float().softmax(dim=1)) / 2
        out.append(p.cpu().numpy())
        if i % 50 == 0:
            print(f"   {i * 128:>6} / 50000   {time.time() - t0:5.0f}s", flush=True)
    return np.concatenate(out)


print("\nfp32:")
p32 = crop_posteriors(half=False)
print("\nfp16:")
p16 = crop_posteriors(half=True)

a32, a16 = p32.argmax(1), p16.argmax(1)
flips = int((a32 != a16).sum())
e32 = 100 * (1 - (a32 == in_labels).mean())
e16 = 100 * (1 - (a16 == in_labels).mean())
print(f"\ntop-1 error   fp32 {e32:.3f}%   fp16 {e16:.3f}%")
print(f"images whose top-1 changed: {flips}  ({100 * flips / 50000:.3f}% of the set)")
np.save("in_crop_fp32.npy", p32.astype(np.float16))

50000 images, 1000 classes

fp32:
        0 / 50000       2s
     6400 / 50000      10s
    12800 / 50000      17s
    19200 / 50000      25s
    25600 / 50000      32s
    32000 / 50000      39s
    38400 / 50000      46s
    44800 / 50000      53s

fp16:
        0 / 50000       2s
     6400 / 50000       9s
    12800 / 50000      16s
    19200 / 50000      23s
    25600 / 50000      30s
    32000 / 50000      37s
    38400 / 50000      44s
    44800 / 50000      51s

top-1 error   fp32 27.642%   fp16 27.640%
images whose top-1 changed: 53  (0.106% of the set)


**53 images** change their top-1 under fp16 — 0.106%, the same size as the effects being measured. And
fp32 took 59s against fp16's 57s, which is noise: the bottleneck is JPEG decoding, not the GPU.

So half precision was buying nothing and would have cost the result. Everything below is fp32.

Before the protocols, the pipeline gets checked against something outside this notebook: torchvision
reports 28.408% top-1 for these weights under centre-crop without flip.

In [22]:
import os

@torch.no_grad()
def imagenet_posteriors(protocol, Q=256, flip=True, grid=5, cache=None):
    """fp32 throughout: the fp16 flip rate (0.106%) is the size of the effects
    being measured, and fp16 was not measurably faster anyway."""
    if cache and os.path.exists(cache):
        print(f"{cache} exists - reusing")
        return np.load(cache)

    tf = T.Compose([T.Resize(Q), T.CenterCrop(224), T.ToTensor(), NORM]) \
         if protocol == "crop" else T.Compose([T.Resize(Q), T.ToTensor(), NORM])
    bs = 128 if protocol == "crop" else 1
    loader = DataLoader(ImageNetVal(OUT, tf), batch_size=bs,
                        num_workers=8, pin_memory=True)

    out, t0 = [], time.time()
    for i, (x, _) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        if protocol == "multicrop":
            H, W = x.shape[2], x.shape[3]
            ys = torch.linspace(0, H - 224, grid).round().long().tolist()
            xs = torch.linspace(0, W - 224, grid).round().long().tolist()
            c = torch.cat([x[:, :, a:a + 224, b:b + 224] for a in ys for b in xs])
            c = torch.cat([c, torch.flip(c, dims=[3])])          # 50 crops
            p = net(c).softmax(dim=1).mean(dim=0, keepdim=True)
        else:
            model = net if protocol == "crop" else dense
            def post(t):
                o = model(t)
                return (o.mean(dim=(2, 3)) if o.dim() == 4 else o).softmax(dim=1)
            p = post(x)
            if flip:
                p = (p + post(torch.flip(x, dims=[3]))) / 2
        out.append(p.cpu().numpy())
        if i % max(1, len(loader) // 12) == 0:
            done = i * bs
            print(f"   {done:>6} / 50000   {time.time() - t0:5.0f}s", flush=True)

    out = np.concatenate(out)
    if cache:
        np.save(cache, out)          # float32: float16 here would undo the point
    return out


# external check first - this should land on torchvision's own reported number
noflip = imagenet_posteriors("crop", flip=False, cache="in_crop_noflip.npy")
e = 100 * (1 - (noflip.argmax(1) == in_labels).mean())
print(f"\ncrop, no flip: {e:.3f}%   torchvision reports 28.408%   gap {e - 28.408:+.3f}")

        0 / 50000       2s
     4096 / 50000       7s
     8192 / 50000      12s
    12288 / 50000      17s
    16384 / 50000      21s
    20480 / 50000      26s
    24576 / 50000      30s
    28672 / 50000      35s
    32768 / 50000      39s
    36864 / 50000      43s
    40960 / 50000      48s
    45056 / 50000      53s
    49152 / 50000      57s

crop, no flip: 28.418%   torchvision reports 28.408%   gap +0.010


**28.418% against torchvision's 28.408%** — five images in fifty thousand. The whole pipeline (data,
labels, normalisation, model) now agrees with something measured outside this notebook.

All four protocols at a single scale, with the Table 5 comparisons fixed in advance.

In [23]:
P = {}
P["crop"]      = imagenet_posteriors("crop",  flip=True, cache="in_crop.npy")
P["dense"]     = imagenet_posteriors("dense", flip=True, cache="in_dense.npy")
P["multicrop"] = imagenet_posteriors("multicrop",        cache="in_multicrop.npy")
P["multicrop & dense"] = (P["multicrop"] + P["dense"]) / 2


def ranks_of(p):
    """Rank of the true class, 1 = top. Gives top-1 and top-5 without a full argsort
    of a 50000 x 1000 array."""
    true = p[np.arange(len(p)), in_labels][:, None]
    return (p > true).sum(1) + 1


R = {k: ranks_of(v) for k, v in P.items()}

print()
for k, r in R.items():
    print(f"{k:<18} top-1 error {100 * (r > 1).mean():6.3f}%   "
          f"top-5 error {100 * (r > 5).mean():6.3f}%")

print("\nPRE-SPECIFIED - the paper's Table 5, written down before looking")
print("6 tests, so the threshold is 0.05 / 6 = 0.0083\n")
for a, b in [("dense", "multicrop"),
             ("dense", "multicrop & dense"),
             ("multicrop", "multicrop & dense")]:
    for k in (1, 5):
        mcnemar(R[a] <= k, R[b] <= k, f"top-{k}  {a} vs {b}")

print("\nEXPLORATORY")
for b in ("dense", "multicrop", "multicrop & dense"):
    for k in (1, 5):
        mcnemar(R["crop"] <= k, R[b] <= k, f"top-{k}  crop vs {b}")

        0 / 50000       2s
     4096 / 50000       7s
     8192 / 50000      11s
    12288 / 50000      17s
    16384 / 50000      21s
    20480 / 50000      26s
    24576 / 50000      30s
    28672 / 50000      35s
    32768 / 50000      39s
    36864 / 50000      44s
    40960 / 50000      49s
    45056 / 50000      54s
    49152 / 50000      58s
        0 / 50000       0s
     4166 / 50000      23s
     8332 / 50000      45s
    12498 / 50000      67s
    16664 / 50000      90s
    20830 / 50000     113s
    24996 / 50000     135s
    29162 / 50000     157s
    33328 / 50000     180s
    37494 / 50000     203s
    41660 / 50000     225s
    45826 / 50000     247s
    49992 / 50000     269s
        0 / 50000       0s
     4166 / 50000     109s
     8332 / 50000     218s
    12498 / 50000     327s
    16664 / 50000     436s
    20830 / 50000     545s
    24996 / 50000     655s
    29162 / 50000     764s
    33328 / 50000     873s
    37494 / 50000     982s
    41660 / 50000    1091s
 

At a single scale the ordering already matches Table 5, and now the tests have the power to say so:
crop → dense → multi-crop → combination, with every comparison against crop at p = 0.0000.

---

## 16. The paper's own protocol

§4.2 averages posteriors over three test scales. Everything above was Q = 256 alone, which is not what
Table 5 reports — so the last run is the full thing: Q = {256, 384, 512}.

In [24]:
import shutil

# reuse the Q=256 runs already on disk instead of repeating 26 minutes of work
for proto, old in [("dense", "in_dense.npy"), ("multicrop", "in_multicrop.npy")]:
    new = f"in_{proto}_q256.npy"
    if os.path.exists(old) and not os.path.exists(new):
        shutil.move(old, new)
        print(f"reusing {old} as {new}")

SCALES = (256, 384, 512)

MS = {}
for proto in ("dense", "multicrop"):
    acc = None
    for Q in SCALES:
        print(f"\n{proto}, Q={Q}")
        p = imagenet_posteriors(proto, Q=Q, flip=(proto == "dense"),
                                cache=f"in_{proto}_q{Q}.npy")
        acc = p.astype(np.float64) if acc is None else acc + p
        del p
    MS[proto] = (acc / len(SCALES)).astype(np.float32)   # paper 4.2: average posteriors
    del acc

MS["multicrop & dense"] = (MS["multicrop"] + MS["dense"]) / 2

RM = {k: ranks_of(v) for k, v in MS.items()}

print("\nMULTI-SCALE, Q in {256, 384, 512}\n")
for k, r in RM.items():
    print(f"{k:<18} top-1 error {100 * (r > 1).mean():6.3f}%   "
          f"top-5 error {100 * (r > 5).mean():6.3f}%")

print("\nPRE-SPECIFIED A - Table 5 at the paper's own protocol")
print("6 tests, threshold 0.05 / 6 = 0.0083\n")
for a, b in [("dense", "multicrop"),
             ("dense", "multicrop & dense"),
             ("multicrop", "multicrop & dense")]:
    for k in (1, 5):
        mcnemar(RM[a] <= k, RM[b] <= k, f"top-{k}  {a} vs {b}")

print("\nPRE-SPECIFIED B - Table 4's claim that scale jittering at test time helps")
print("2 tests, threshold 0.05 / 2 = 0.025\n")
for proto in ("dense", "multicrop"):
    mcnemar(R[proto] <= 1, RM[proto] <= 1, f"top-1  {proto}: single Q=256 vs three scales")

reusing in_dense.npy as in_dense_q256.npy
reusing in_multicrop.npy as in_multicrop_q256.npy

dense, Q=256
in_dense_q256.npy exists - reusing

dense, Q=384
        0 / 50000       0s
     4166 / 50000      34s
     8332 / 50000      64s
    12498 / 50000      94s
    16664 / 50000     125s
    20830 / 50000     156s
    24996 / 50000     186s
    29162 / 50000     217s
    33328 / 50000     247s
    37494 / 50000     277s
    41660 / 50000     307s
    45826 / 50000     337s
    49992 / 50000     367s

dense, Q=512
        0 / 50000       0s
     4166 / 50000      50s
     8332 / 50000      95s
    12498 / 50000     141s
    16664 / 50000     188s
    20830 / 50000     235s
    24996 / 50000     282s
    29162 / 50000     329s
    33328 / 50000     376s
    37494 / 50000     422s
    41660 / 50000     469s
    45826 / 50000     515s
    49992 / 50000     561s

multicrop, Q=256
in_multicrop_q256.npy exists - reusing

multicrop, Q=384
        0 / 50000       0s
     4166 / 50000     110s


**24.812% / 7.420%** for the combination, against the paper's 24.4 / 7.2.

The ordering reproduces and so do the gaps — dense to the combination is 0.430 here against the
paper's 0.4. The ~0.45 point offset is the weights, not the protocol: it barely moves across the three
(0.442, 0.500, 0.412), which makes protocol quality and weight quality additive rather than
interacting.

Both pre-specified families pass. And the two levers can finally be put side by side: averaging over
three test scales is worth **1.01–1.13 points**, while choosing multi-crop over dense is worth
**0.142** and does not reach significance. The paper gives §4.3 and a table to the smaller of the two.

---

## What this reproduced, and what it did not

**Reproduced** — Table 1 and Table 2 for all six configurations; §3.2's conversion to float32
round-off; §3.1's pre-initialisation claim; Table 5's ordering and gaps on the paper's own validation
set; Table 4's claim that test-time scale jittering helps.

**Not reproduced** — no network here was trained to convergence on ImageNet; the paper's own
initialisation does not train configuration A, so the donor its procedure requires could not be
produced; Table 4's Q = {S−32, S, S+32} finding needs a training scale S that these weights do not
have; §4.4 ConvNet fusion was not attempted.

**Measured here and not in the paper** — what A-LRN costs in time and memory; milliseconds per GMAC
for each step up Table 1; that the evaluation protocol's benefit is nearly independent of weight
quality; that the scale lever is about three times the protocol lever; and that dense evaluation
improves the candidate list rather than the top slot.